# Fit 2 — $N_f=4$ free constant plus running series, order 5, no priors

Wilson flow, on-the-fly TLN, chiral and infinite-volume limits, and the same downstream analysis used in the Fit 4 notebooks. The interpolation ansatz is
$$
\beta(x)=p_0+x^2\sum_{n=1}^{5}p_n x^{n-1}, \qquad x=g_{\rm GF}^2.
$$

The coefficients $p_0,\ldots,p_5$ are fitted without priors. Outputs are organized as `fit2/order_5_nopriors/t_<min>_<max>/<mode>/`, where mode is `diagonal` or `correlated`.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
import copy, gc, json, sys

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').is_file():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Run this notebook from inside the ContinuousBetaFunction repository')
sys.path.insert(0, str(REPO_ROOT / 'src'))

import betafn
import gvar as gv
import lsqfit
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from IPython.display import display

print('betafn package:', Path(betafn.__file__).resolve())
print('repository:', REPO_ROOT)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'text.usetex': True, 'font.family': 'serif',
    'font.serif': ['Computer Modern'], 'font.size': 15,
    'figure.dpi': 300, 'savefig.dpi': 300,
    'axes.labelsize': 20, 'axes.titlesize': 18,
    'legend.fontsize': 11, 'xtick.labelsize': 18, 'ytick.labelsize': 18,
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
    'grid.alpha': .35, 'grid.linestyle': '-', 'axes.axisbelow': True,
})

## Configuration and complete window catalogue

Change physics/statistical choices only here. Every downstream cell reads these values.

In [ ]:
FIT_ID = 'fit2'
ORDER = 5
FIT_NO_PRIORS = True
FIT_WIDTH = None
FIT_PRIOR_COUNT = 0
FIT_WATERMARK = rf'{FIT_ID}, free $p_0$ + $x^2$ series through $p_{{{ORDER}}}$, no priors'
FIT_FOOTER = 'no coefficient priors'
CORRECTION = 'tln'
DATA_DIR = Path('/Users/yaman/Contbetafn/data/New Four')
OUTPUT_ROOT = REPO_ROOT / 'src/betafn' / 'fit2' / f'order_{ORDER}_nopriors'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# The 7p125 entry is intentionally configured before its files arrive.
# Dataset discovery skips it now and includes it automatically once the three
# finite-mass ensembles are present on each configured volume.
STRONG_COUPLINGS = ('7p00', '7p125', '7p25', '7p50')
WEAK_COUPLINGS = (
    '8p00', '8p50', '9p00', '9p50', '10p0', '11p0',
    '12p0', '14p0', '16p0', '18p0', '20p0',
)
COUPLINGS = STRONG_COUPLINGS + WEAK_COUPLINGS
STRONG_VOLUMES = ('l20l20l20t40', 'l24l24l24t48', 'l32l32l32t64')
VOLUMES = {
    **{coupling: STRONG_VOLUMES for coupling in STRONG_COUPLINGS},
    '8p00': (
        'l32l32l32t64', 'l40l40l40t80', 'l48l48l48t96',
    ),
    **{
        coupling: ('l32l32l32t64', 'l40l40l40t80', 'l48l48l48t96')
        for coupling in WEAK_COUPLINGS if coupling != '8p00'
    },
}
FLOW = 'wilson'
OBSERVABLES = ('p', 'c', 's')
OP_LABELS = {'p': 'Wilson', 'c': 'clover', 's': 'Symanzik'}
# Match the operator convention used in the reference scan notebook.
OP_COLORS = {
    'p': plt.cm.YlOrRd(0.75),
    'c': plt.cm.YlGn(0.75),
    's': plt.cm.Greys(0.75),
}

# Exhaustive integer windows used by the scan reference notebook.
TMIN_VALUES = tuple(range(5, 6))
TMAX_LIMIT = 6
WINDOWS = tuple(
    (float(tmin), float(tmax))
    for tmin in TMIN_VALUES
    for tmax in range(tmin + 1, TMAX_LIMIT + 1)
)
CENTRAL_WINDOW = (5.0, 6.0)
# Extend through the t0-defining coupling, g_GF^2(t0) ~= 15.79.
# Domain checks in cnt_xtrp still prevent unsupported extrapolation.
G2_GRID = (0.9, 16.0, 0.1)
TARGET_G2 = (1.0, 1.5, 3.0, 5.0, 8.0, 10.0, 12.0, 15.8)
# t0^2 <E(t0)> = 0.3 and g_GF^2 = (16 pi^2 / 3) t^2 <E>.
G2_T0 = (8.0 / 5.0) * np.pi**2
PLOT_MSBAR_LAMBDA = False
POLYNOMIAL_PARAMETER_INDICES = tuple(range(ORDER + 1))

def window_tag(window):
    return f't_{window[0]:g}_{window[1]:g}'.replace('.', 'p')

def case_dir(window, mode=None):
    path = OUTPUT_ROOT / window_tag(window)
    if mode is not None:
        path = path / mode
    path.mkdir(parents=True, exist_ok=True)
    return path

def add_fit_footer(fig):
    fig.text(0.995, 0.005, FIT_FOOTER, ha='right', va='bottom',
             fontsize=8, color='gray', alpha=.75)

def save_figure(fig, window, name, mode=None):
    base = case_dir(window, mode) / name
    add_fit_footer(fig)
    fig.savefig(base.with_suffix('.png'), dpi=300, bbox_inches='tight')
    fig.savefig(base.with_suffix('.pdf'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

def beta_value(coupling):
    return float(coupling.replace('p', '.'))

def volume_value(volume):
    return np.prod([float(x) for x in volume.replace('t', 'l').split('l')[1:] if x])


def coupling_figure(couplings, ncols=5):
    nrows = max(1, int(np.ceil(len(couplings) / ncols)))
    return plt.subplots(
        nrows, ncols,
        figsize=(4.4 * ncols, 4.2 * nrows),
        squeeze=False,
    )

def available_flow_times(window):
    return [t for t in sorted(bf.ntrp_fits[FLOW][OBSERVABLES[0]], key=float)
            if window[0] <= float(t) <= window[1]]

def pt_over_g4(x, loops):
    x = np.asarray(x, dtype=float)
    pt = bf.perturbative_beta_function
    return sum(-coefficient * x**power / pt.nrm**(power + 1)
               for power, coefficient in enumerate(pt.b[:loops]))

print(f'{len(WINDOWS)} windows:', WINDOWS)
print('output root:', OUTPUT_ROOT)

## Analysis construction

The pulled upstream package computes TLN on demand. No external `.tln` path is supplied. The raw `Es` history is the rectangle measurement and is replaced by `(5/3)Ep-(2/3)Es` before averaging and TLN.

In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(DATA_DIR)

bf = betafn.BetaFunction(nf=4)

def free_constant_running_interpolation(x, p):
    x = np.asarray(x)
    running = sum(
        p[f'p{power}'][0] * x**(power - 1)
        for power in range(1, ORDER + 1)
    )
    return p['p0'][0] + x**2 * running

interpolation = betafn.InterpolationSpec(
    fcn=free_constant_running_interpolation,
    prior=None,
    p0={f'p{power}': [0.0] for power in POLYNOMIAL_PARAMETER_INDICES},
    xerrors=False,
)

config = betafn.AnalysisConfig(
    data_path=str(DATA_DIR),
    interpolation=interpolation,
    continuum_window=CENTRAL_WINDOW,
    g2_grid=G2_GRID,
    flows=(FLOW,),
    observables=OBSERVABLES,
    combine={'s': {'p': 5.0 / 3.0, 's': -2.0 / 3.0}},
    couplings=COUPLINGS,
    volumes=VOLUMES,
    correction=CORRECTION,
    binsize=15,
    use_gamma_method=True,
    process_window=(1.5, 10.5),
    fit_window=(2.0, 10.0),
    error_mode='fit',
    cov_mode='kernel',
    correlated=True,
    verbosity=0,
)
display(pd.DataFrame([config.describe()]).T.rename(columns={0: 'fit2 setting'}))

## Stage 1 — processing with TLN

In [ ]:
bf.run_processing(config)
print('correction:', bf._process_config.correction)
print('processed couplings:', sorted(bf.avg_data, key=beta_value))
print('binsize:', bf._binsize)
print(bf.data_report())

## Plot 1 — processed largest-volume data

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for operator in OBSERVABLES:
    first = True
    for coupling in sorted(bf.avg_data, key=beta_value):
        massless_volumes = [
            volume for volume in bf.avg_data[coupling]
            if '0p00' in bf.avg_data[coupling][volume]
        ]
        if not massless_volumes:
            continue
        volume = max(massless_volumes, key=volume_value)
        _, g2 = bf.processed_series(coupling, volume, '0p00', FLOW, operator, 'g2')
        _, beta = bf.processed_series(coupling, volume, '0p00', FLOW, operator, 'beta')
        ratio = np.asarray([b / x**2 for x, b in zip(g2, beta)], dtype=object)
        ax.errorbar(gv.mean(g2), gv.mean(ratio), yerr=gv.sdev(ratio), fmt='o', ms=2.5,
                    alpha=.6, color=OP_COLORS[operator],
                    label=OP_LABELS[operator] if first else None)
        first = False
ax.set(xlabel=r'$g^2_{GF}$', ylabel=r'$\beta_{GF}/g_{GF}^4$', title='Directly simulated massless largest-volume data (TLN)')
ax.legend(frameon=False)
base = OUTPUT_ROOT / 'processed_largest_volume_fit2'
add_fit_footer(fig)
fig.savefig(base.with_suffix('.png'), dpi=300, bbox_inches='tight')
fig.savefig(base.with_suffix('.pdf'), dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)

## Stages 2–4 — chiral, infinite volume, and fixed fit2 interpolation

In [ ]:
bf.run_chiral(config)
bf.run_infinite_volume(config)
bf.run_interpolation(config)
print(bf.stage_summary('chiral'))
print(bf.stage_summary('infinite_volume'))
print(bf.stage_summary('interpolation'))

## Chiral extrapolation diagnostics

For each strong-coupling ensemble, the existing chiral stage fits $g_{\mathrm{GF}}^2$ and $\beta_{\mathrm{GF}}$ separately at fixed $(\beta_b,L/a,t/a^2)$ with the linear form

$$
X(m)=X(0)+k_m m,\qquad X\in\{g_{\mathrm{GF}}^2,\beta_{\mathrm{GF}}\}.
$$

The three simulated masses are used without coefficient priors. The fitted intercept $X(0)$ is inserted as the `0p00` datum before the infinite-volume and interpolation stages. The plots below show the input masses, fitted curves, and massless intercepts for representative flow times; the CSV records every chiral fit. The configured $\beta_b=7.125$ ensembles are picked up automatically when their files are added.


In [ ]:
ACTIVE_COUPLINGS = tuple(sorted(bf.avg_data, key=beta_value))
ACTIVE_STRONG_COUPLINGS = tuple(
    coupling for coupling in STRONG_COUPLINGS if coupling in bf.avg_data
)
MISSING_STRONG_COUPLINGS = tuple(
    coupling for coupling in STRONG_COUPLINGS if coupling not in bf.avg_data
)
IV_COUPLINGS = tuple(
    coupling for coupling in ACTIVE_COUPLINGS
    if coupling in bf.iv_fits
    and any(bf.iv_fits[coupling]['g2'][FLOW][operator]
            for operator in OBSERVABLES)
)

print('active couplings:', ACTIVE_COUPLINGS)
print('strong couplings awaiting data:', MISSING_STRONG_COUPLINGS or 'none')

EXPECTED_CHIRAL_MASSES = ('0p001', '0p0025', '0p005')
inventory_rows = []
for coupling in STRONG_COUPLINGS:
    if coupling not in bf.avg_data:
        inventory_rows.append({
            'coupling': coupling,
            'beta_b': beta_value(coupling),
            'volume': 'all configured volumes',
            'finite_masses': '',
            'massless_limit_available': False,
            'status': 'awaiting files',
        })
        continue
    for volume in VOLUMES[coupling]:
        volume_data = bf.avg_data[coupling].get(volume, {})
        finite_masses = sorted(
            (mass for mass in volume_data if mass != '0p00'),
            key=lambda mass: float(mass.replace('p', '.')),
        )
        inventory_rows.append({
            'coupling': coupling,
            'beta_b': beta_value(coupling),
            'volume': volume,
            'finite_masses': ', '.join(finite_masses),
            'massless_limit_available': '0p00' in volume_data,
            'status': (
                'ready'
                if tuple(finite_masses) == EXPECTED_CHIRAL_MASSES
                else 'incomplete mass set'
            ),
        })

chiral_inventory = pd.DataFrame(inventory_rows)
chiral_inventory.to_csv(OUTPUT_ROOT / 'chiral_ensemble_inventory.csv', index=False)
display(chiral_inventory)

chiral_quality_rows = []
for coupling in ACTIVE_STRONG_COUPLINGS:
    for volume in VOLUMES[coupling]:
        if volume not in bf.ch_fits.get(coupling, {}):
            continue
        for quantity in ('g2', 'beta'):
            for operator in OBSERVABLES:
                fits = bf.ch_fits[coupling][volume][quantity][FLOW][operator]
                qualities = bf.ch_qof[coupling][volume][quantity][FLOW][operator]
                for time, params in fits.items():
                    qof = qualities[time]
                    intercept = bf.chiral.model.evaluate(0.0, params)
                    chiral_quality_rows.append({
                        'coupling': coupling,
                        'beta_b': beta_value(coupling),
                        'volume': volume,
                        'quantity': quantity,
                        'operator': operator,
                        'flow_time': float(time),
                        'intercept_mean': float(gv.mean(intercept)),
                        'intercept_sdev': float(gv.sdev(intercept)),
                        'chi2': float(qof['chi2']),
                        'dof': int(qof['dof']),
                        'p_value': float(qof['p-value']),
                    })

chiral_fit_quality = pd.DataFrame(chiral_quality_rows)
chiral_fit_quality.to_csv(
    OUTPUT_ROOT / 'chiral_fit_quality_summary.csv', index=False
)
display(chiral_fit_quality)

CHIRAL_PLOT_TIMES = (4.0, 6.0, 8.0)
CHIRAL_TIME_COLORS = dict(zip(
    CHIRAL_PLOT_TIMES,
    plt.cm.viridis(np.linspace(0.15, 0.85, len(CHIRAL_PLOT_TIMES))),
))

def plot_chiral_extrapolations(quantity):
    ylabel = (
        r'$g^2_{\mathrm{GF}}$'
        if quantity == 'g2'
        else r'$\beta_{\mathrm{GF}}$'
    )
    for coupling in ACTIVE_STRONG_COUPLINGS:
        volumes = [
            volume for volume in VOLUMES[coupling]
            if volume in bf.ch_fits.get(coupling, {})
            and any(
                bf.ch_fits[coupling][volume]['g2'][FLOW][operator]
                for operator in OBSERVABLES
            )
        ]
        if not volumes:
            continue
        fig, axes = plt.subplots(
            len(volumes), len(OBSERVABLES),
            figsize=(15, 4.2 * len(volumes)),
            squeeze=False,
        )
        for row, volume in enumerate(volumes):
            for column, operator in enumerate(OBSERVABLES):
                ax = axes[row, column]
                available = sorted(
                    bf.ch_fits[coupling][volume][quantity][FLOW][operator],
                    key=float,
                )
                selected = []
                selected_times = set()
                for target in CHIRAL_PLOT_TIMES:
                    if not available:
                        continue
                    time = min(available, key=lambda value: abs(float(value) - target))
                    if time not in selected_times:
                        selected.append((target, time))
                        selected_times.add(time)

                pvalue_lines = []
                for target, time in selected:
                    data = bf.chiral.fetch(
                        'inputs', (coupling, volume, quantity, FLOW, operator, time)
                    )
                    params = bf.chiral.fetch(
                        'fits', (coupling, volume, quantity, FLOW, operator, time)
                    )
                    qof = bf.chiral.fetch(
                        'quality', (coupling, volume, quantity, FLOW, operator, time)
                    )
                    masses = np.asarray(data.x, dtype=float)
                    values = np.asarray(data.y, dtype=object)
                    mass_grid = np.linspace(0.0, 1.05 * masses.max(), 160)
                    curve = np.asarray([
                        bf.chiral.model.evaluate(mass, params)
                        for mass in mass_grid
                    ], dtype=object)
                    intercept = bf.chiral.model.evaluate(0.0, params)
                    color = CHIRAL_TIME_COLORS[target]
                    label = rf'$t/a^2={float(time):g}$'
                    ax.errorbar(
                        masses, gv.mean(values), yerr=gv.sdev(values),
                        fmt='o', ms=4, capsize=3, color=color, label=label,
                    )
                    ax.plot(mass_grid, gv.mean(curve), color=color, lw=1.5)
                    ax.fill_between(
                        mass_grid,
                        gv.mean(curve) - gv.sdev(curve),
                        gv.mean(curve) + gv.sdev(curve),
                        color=color, alpha=.14, linewidth=0,
                    )
                    ax.errorbar(
                        [0.0], [gv.mean(intercept)], yerr=[gv.sdev(intercept)],
                        fmt='s', ms=5, capsize=3, color=color,
                        markeredgecolor='black', markeredgewidth=.4,
                    )
                    pvalue_lines.append(
                        rf'$t={float(time):g}$: $p={100.0 * float(qof["p-value"]):.1f}\%$'
                    )

                spatial_extent = int(volume.split('l')[1])
                ax.set_xlabel(r'$am_f$')
                ax.text(
                    .97, .97,
                    rf'$\beta_b={beta_value(coupling):g}$, ' + OP_LABELS[operator]
                    + '\n' + rf'$L/a={spatial_extent}$',
                    transform=ax.transAxes, ha='right', va='top', fontsize=10.5,
                )
                ax.text(
                    .03, .97, '\n'.join(pvalue_lines), transform=ax.transAxes,
                    ha='left', va='top', fontsize=8.5,
                )

        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, ncol=max(1, len(handles)), loc='upper center', frameon=False)
        fig.supylabel(ylabel)
        fig.tight_layout(rect=(.03, .02, 1, .93))
        base = OUTPUT_ROOT / f'chiral_extrapolation_{quantity}_{coupling}_fit2'
        add_fit_footer(fig)
        fig.savefig(base.with_suffix('.png'), dpi=300, bbox_inches='tight')
        fig.savefig(base.with_suffix('.pdf'), dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(fig)

plot_chiral_extrapolations('g2')
plot_chiral_extrapolations('beta')

# Confirm visually that the extrapolated strong-coupling points join the
# directly simulated massless weak-coupling data used downstream.
fig, ax = plt.subplots(figsize=(9, 6.5))
for operator in OBSERVABLES:
    first = True
    for coupling in ACTIVE_COUPLINGS:
        massless_volumes = [
            volume for volume in bf.avg_data[coupling]
            if '0p00' in bf.avg_data[coupling][volume]
        ]
        if not massless_volumes:
            continue
        volume = max(massless_volumes, key=volume_value)
        _, g2 = bf.processed_series(
            coupling, volume, '0p00', FLOW, operator, 'g2'
        )
        _, beta = bf.processed_series(
            coupling, volume, '0p00', FLOW, operator, 'beta'
        )
        ratio = np.asarray(beta, dtype=object) / np.asarray(g2, dtype=object)**2
        ax.errorbar(
            gv.mean(g2), gv.mean(ratio), yerr=gv.sdev(ratio),
            fmt='o', ms=2.5, alpha=.6, color=OP_COLORS[operator],
            label=OP_LABELS[operator] if first else None,
        )
        first = False
ax.set(
    xlabel=r'$g^2_{\mathrm{GF}}$',
    ylabel=r'$\beta_{\mathrm{GF}}/g_{\mathrm{GF}}^4$',
    title='Massless-limit largest-volume data after chiral extrapolation',
)
ax.legend(frameon=False)
base = OUTPUT_ROOT / 'processed_massless_limit_largest_volume_fit2'
add_fit_footer(fig)
fig.savefig(base.with_suffix('.png'), dpi=300, bbox_inches='tight')
fig.savefig(base.with_suffix('.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

# Raw massless infinite-volume intercepts before interpolation.
fig, axes = plt.subplots(1, 2, figsize=(15, 6.2), sharex=True)
ordered_iv_couplings = tuple(
    sorted(IV_COUPLINGS, key=beta_value, reverse=True)
)
qualitative_colors = tuple(plt.get_cmap('tab20').colors)
beta_colors = {
    coupling: qualitative_colors[index % len(qualitative_colors)]
    for index, coupling in enumerate(ordered_iv_couplings)
}
operator_markers = {'p': 'o', 'c': 's', 's': '^'}

for coupling in ordered_iv_couplings:
    color = beta_colors[coupling]
    for operator in OBSERVABLES:
        g2_fits = bf.iv_fits[coupling]['g2'][FLOW][operator]
        beta_fits = bf.iv_fits[coupling]['beta'][FLOW][operator]
        common_times = sorted(set(g2_fits) & set(beta_fits), key=float)
        for time in common_times:
            g2_iv = bf.infinite_volume.model.evaluate(0.0, g2_fits[time])
            beta_iv = bf.infinite_volume.model.evaluate(0.0, beta_fits[time])
            ratio_iv = beta_iv / g2_iv**2
            style = dict(
                fmt=operator_markers[operator], ms=3.8, capsize=2,
                color=color, alpha=.58, linestyle='none',
            )
            axes[0].errorbar(
                gv.mean(g2_iv), gv.mean(beta_iv),
                xerr=gv.sdev(g2_iv), yerr=gv.sdev(beta_iv), **style,
            )
            axes[1].errorbar(
                gv.mean(g2_iv), gv.mean(ratio_iv),
                xerr=gv.sdev(g2_iv), yerr=gv.sdev(ratio_iv), **style,
            )

axes[0].set(
    xlabel=r'$g^2_{\mathrm{GF}}$', ylabel=r'$\beta_{\mathrm{GF}}$',
    title=r'Raw massless IV data: $\beta_{\mathrm{GF}}$',
)
axes[1].set(
    xlabel=r'$g^2_{\mathrm{GF}}$',
    ylabel=r'$\beta_{\mathrm{GF}}/g_{\mathrm{GF}}^4$',
    title=r'Raw massless IV data: $\beta_{\mathrm{GF}}/g_{\mathrm{GF}}^4$',
)
plot_xmax = max(G2_GRID[1], G2_T0) + 0.25
xp_pt = np.linspace(0.001, plot_xmax, 900)
pt_styles = (
    (1, ':', '1-loop'),
    (2, '-.', '2-loop'),
    (3, '-', '3-loop'),
)
for loops, linestyle, label in pt_styles:
    ratio_pt = pt_over_g4(xp_pt, loops)
    axes[0].plot(
        xp_pt, xp_pt**2 * ratio_pt, color='gray', ls=linestyle,
        lw=1.5, zorder=1,
    )
    axes[1].plot(
        xp_pt, ratio_pt, color='gray', ls=linestyle,
        lw=1.5, zorder=1,
    )
for ax in axes:
    ax.set_xlim(0.0, plot_xmax)
    ax.axvline(G2_T0, color='black', linestyle='--', lw=1.5, zorder=1)
axes[0].set_ylim(-13.0, 0.5)
operator_handles = [
    Line2D(
        [0], [0], marker=operator_markers[operator], linestyle='none',
        color='gray', markeredgecolor='black', markeredgewidth=.4,
        label=OP_LABELS[operator],
    )
    for operator in OBSERVABLES
]
coupling_handles = [
    Line2D(
        [0], [0], marker='o', linestyle='none',
        color=beta_colors[coupling], markersize=6,
        label=rf'$\beta_b={beta_value(coupling):g}$',
    )
    for coupling in ordered_iv_couplings
]
pt_handles = [
    Line2D([0], [0], color='gray', linestyle=linestyle, lw=1.5, label=label)
    for _, linestyle, label in pt_styles
]
t0_handle = Line2D(
    [0], [0], color='black', linestyle='--', lw=1.5,
    label=rf'$g^2_{{\mathrm{{GF}}}}(t_0)=\frac{{8}}{{5}}\pi^2={G2_T0:.9f}$',
)
axes[0].legend(
    handles=operator_handles + pt_handles + [t0_handle],
    frameon=False, fontsize=8.5, ncol=2,
)
axes[1].legend(
    handles=coupling_handles, frameon=False, fontsize=8,
    ncol=2, title=r'bare coupling',
)
fig.suptitle(
    r'Massless infinite-volume data before interpolation '    r'($\beta_b=20$ through the strongest available coupling)',
)
fig.tight_layout(rect=(0, 0, 1, .94))
base = OUTPUT_ROOT / f'massless_infinite_volume_beta_and_ratio_vs_g2_{FIT_ID}'
add_fit_footer(fig)
fig.savefig(base.with_suffix('.png'), dpi=300, bbox_inches='tight')
fig.savefig(base.with_suffix('.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)


## Plot 2 — infinite-volume extrapolations for every bare coupling

In [ ]:
fig, axes = coupling_figure(IV_COUPLINGS)
for ax, coupling in zip(axes.ravel(), IV_COUPLINGS):
    times = sorted(bf.iv_fits[coupling]['g2'][FLOW][OBSERVABLES[0]], key=float)
    time = min(times, key=lambda t: abs(float(t) - CENTRAL_WINDOW[0]))
    for operator in OBSERVABLES:
        xfit, yfit, data = bf.infinite_volume_curve(coupling, FLOW, operator, time, x='g2')
        intercept = bf.infinite_volume.model.evaluate(0.0, bf.iv_fits[coupling]['g2'][FLOW][operator][time])
        color = OP_COLORS[operator]
        ax.errorbar(gv.mean(data.x), gv.mean(data.y), xerr=gv.sdev(data.x), yerr=gv.sdev(data.y),
                    fmt='o', capsize=3, color=color, label=OP_LABELS[operator])
        ax.plot(xfit, gv.mean(yfit), color=color)
        ax.fill_between(xfit, gv.mean(yfit)-gv.sdev(yfit), gv.mean(yfit)+gv.sdev(yfit), color=color, alpha=.15)
        ax.errorbar([0], [gv.mean(intercept)], yerr=[gv.sdev(intercept)], fmt='s', capsize=3, color=color)
    ax.set(title=rf'$\beta_b={beta_value(coupling):g}$, $t/a^2={time}$', xlabel=r'$1/V$', ylabel=r'$g^2_{GF}$')
for ax in axes.ravel()[len(IV_COUPLINGS):]:
    ax.set_visible(False)

handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=3, loc='upper center', frameon=False)
fig.tight_layout(rect=(0, 0, 1, .96))
base = OUTPUT_ROOT / 'infinite_volume_all_beta_fit2'
add_fit_footer(fig)
fig.savefig(base.with_suffix('.png'), dpi=300, bbox_inches='tight')
fig.savefig(base.with_suffix('.pdf'), dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)

In [ ]:
# Infinite-volume extrapolation: beta_GF versus 1/V

fig, axes = coupling_figure(IV_COUPLINGS)

for ax, coupling in zip(axes.ravel(), IV_COUPLINGS):
  available_times = sorted(
      bf.iv_fits[coupling]['beta'][FLOW][OBSERVABLES[0]],
      key=float,
  )
  time = min(
      available_times,
      key=lambda value: abs(float(value) - CENTRAL_WINDOW[0]),
  )

  for operator in OBSERVABLES:
      x_fit, beta_fit, beta_data = bf.infinite_volume_curve(
          coupling,
          FLOW,
          operator,
          time,
          x='beta',
      )

      x_fit = np.asarray(x_fit, dtype=float)
      beta_fit = np.asarray(beta_fit, dtype=object)
      beta_data_y = np.asarray(beta_data.y, dtype=object)

      color = OP_COLORS[operator]

      # Finite-volume beta_GF data.
      ax.errorbar(
          beta_data.x,
          gv.mean(beta_data_y),
          yerr=gv.sdev(beta_data_y),
          fmt='o',
          ms=5,
          capsize=3,
          elinewidth=1.0,
          color=color,
          label=OP_LABELS[operator],
          zorder=3,
      )

      # Infinite-volume extrapolation curve.
      ax.plot(
          x_fit,
          gv.mean(beta_fit),
          color=color,
          lw=1.6,
          zorder=2,
      )

      ax.fill_between(
          x_fit,
          gv.mean(beta_fit) - gv.sdev(beta_fit),
          gv.mean(beta_fit) + gv.sdev(beta_fit),
          color=color,
          alpha=0.16,
          linewidth=0,
          zorder=1,
      )

      # Infinite-volume intercept at 1/V = 0.
      beta_params = bf.iv_fits[coupling]['beta'][FLOW][operator][time]
      beta_infinite = bf.infinite_volume.model.evaluate(
          0.0,
          beta_params,
      )

      ax.errorbar(
          [0.0],
          [gv.mean(beta_infinite)],
          yerr=[gv.sdev(beta_infinite)],
          fmt='s',
          ms=6,
          capsize=3,
          color=color,
          markeredgecolor='black',
          markeredgewidth=0.4,
          zorder=4,
      )

  ax.set_title(
      rf'$\beta_b={beta_value(coupling):g}$, '
      rf'$t/a^2={float(time):g}$'
  )
  ax.set_xlabel(r'$1/V$')
  ax.set_ylabel(r'$\beta_{\mathrm{GF}}$')
  ax.minorticks_on()
  ax.grid(which='major', linestyle='-', alpha=0.35)
  ax.grid(which='minor', linestyle='--', alpha=0.20)

for ax in axes.ravel()[len(IV_COUPLINGS):]:
    ax.set_visible(False)

# Shared legend without duplicate entries.
handles, labels = axes.ravel()[0].get_legend_handles_labels()
unique = dict(zip(labels, handles))

fig.legend(
  unique.values(),
  unique.keys(),
  ncol=len(OBSERVABLES),
  loc='upper center',
  frameon=False,
)

fig.suptitle(
  r'Infinite-volume extrapolation of $\beta_{\mathrm{GF}}$',
  y=0.995,
)

fig.tight_layout(rect=(0, 0, 1, 0.95))

output_base = OUTPUT_ROOT / f'infinite_volume_beta_vs_invV_{FIT_ID}'
add_fit_footer(fig)
fig.savefig(
  output_base.with_suffix('.png'),
  dpi=300,
  bbox_inches='tight',
)
fig.savefig(
  output_base.with_suffix('.pdf'),
  dpi=300,
  bbox_inches='tight',
)

plt.show()
plt.close(fig)


In [ ]:
# Infinite-volume extrapolation:
# beta_GF / g_GF^4 versus 1/V

fig, axes = coupling_figure(IV_COUPLINGS)

for ax, coupling in zip(axes.ravel(), IV_COUPLINGS):
  available_times = sorted(
      bf.iv_fits[coupling]['beta'][FLOW][OBSERVABLES[0]],
      key=float,
  )
  time = min(
      available_times,
      key=lambda value: abs(float(value) - CENTRAL_WINDOW[0]),
  )

  for operator in OBSERVABLES:
      # Retrieve the beta_GF and g_GF^2 data used in the IV fits.
      _, _, beta_data = bf.infinite_volume_curve(
          coupling,
          FLOW,
          operator,
          time,
          x='beta',
      )
      _, _, g2_data = bf.infinite_volume_curve(
          coupling,
          FLOW,
          operator,
          time,
          x='g2',
      )

      # Confirm that beta and g^2 use the same volumes in the same order.
      if beta_data.labels != g2_data.labels:
          raise RuntimeError(
              f'Volume ordering mismatch for {coupling}, '
              f'{operator}, t/a^2={time}'
          )

      inv_volume = np.asarray(beta_data.x, dtype=float)
      beta_values = np.asarray(beta_data.y, dtype=object)
      g2_values = np.asarray(g2_data.y, dtype=object)

      # Since g2_values = g_GF^2, g_GF^4 = g2_values**2.
      ratio_data = beta_values / g2_values**2

      beta_params = bf.iv_fits[coupling]['beta'][FLOW][operator][time]
      g2_params = bf.iv_fits[coupling]['g2'][FLOW][operator][time]

      x_fit = np.linspace(
          float(np.min(inv_volume)),
          float(np.max(inv_volume)),
          300,
      )

      beta_fit = np.asarray(
          [
              bf.infinite_volume.model.evaluate(x, beta_params)
              for x in x_fit
          ],
          dtype=object,
      )
      g2_fit = np.asarray(
          [
              bf.infinite_volume.model.evaluate(x, g2_params)
              for x in x_fit
          ],
          dtype=object,
      )

      ratio_fit = beta_fit / g2_fit**2
      color = OP_COLORS[operator]

      # Finite-volume beta_GF/g_GF^4 data.
      ax.errorbar(
          inv_volume,
          gv.mean(ratio_data),
          yerr=gv.sdev(ratio_data),
          fmt='o',
          ms=5,
          capsize=3,
          elinewidth=1.0,
          color=color,
          label=OP_LABELS[operator],
          zorder=3,
      )

      # Ratio of the fitted beta_GF and g_GF^2 IV curves.
      ax.plot(
          x_fit,
          gv.mean(ratio_fit),
          color=color,
          lw=1.6,
          zorder=2,
      )

      ax.fill_between(
          x_fit,
          gv.mean(ratio_fit) - gv.sdev(ratio_fit),
          gv.mean(ratio_fit) + gv.sdev(ratio_fit),
          color=color,
          alpha=0.16,
          linewidth=0,
          zorder=1,
      )

      # Infinite-volume ratio at 1/V = 0.
      beta_infinite = bf.infinite_volume.model.evaluate(
          0.0,
          beta_params,
      )
      g2_infinite = bf.infinite_volume.model.evaluate(
          0.0,
          g2_params,
      )
      ratio_infinite = beta_infinite / g2_infinite**2

      ax.errorbar(
          [0.0],
          [gv.mean(ratio_infinite)],
          yerr=[gv.sdev(ratio_infinite)],
          fmt='s',
          ms=6,
          capsize=3,
          color=color,
          markeredgecolor='black',
          markeredgewidth=0.4,
          zorder=4,
      )

  ax.set_title(
      rf'$\beta_b={beta_value(coupling):g}$, '
      rf'$t/a^2={float(time):g}$'
  )
  ax.set_xlabel(r'$1/V$')
  ax.set_ylabel(r'$\beta_{\mathrm{GF}}/g_{\mathrm{GF}}^4$')
  ax.minorticks_on()
  ax.grid(which='major', linestyle='-', alpha=0.35)
  ax.grid(which='minor', linestyle='--', alpha=0.20)

for ax in axes.ravel()[len(IV_COUPLINGS):]:
    ax.set_visible(False)

# Shared legend without duplicate entries.
handles, labels = axes.ravel()[0].get_legend_handles_labels()
unique = dict(zip(labels, handles))

fig.legend(
  unique.values(),
  unique.keys(),
  ncol=len(OBSERVABLES),
  loc='upper center',
  frameon=False,
)

fig.suptitle(
  r'Infinite-volume extrapolation of '
  r'$\beta_{\mathrm{GF}}/g_{\mathrm{GF}}^4$',
  y=0.995,
)

fig.tight_layout(rect=(0, 0, 1, 0.95))

output_base = (
  OUTPUT_ROOT
  / f'infinite_volume_beta_over_g4_vs_invV_{FIT_ID}'
)

add_fit_footer(fig)
fig.savefig(
  output_base.with_suffix('.png'),
  dpi=300,
  bbox_inches='tight',
)
fig.savefig(
  output_base.with_suffix('.pdf'),
  dpi=300,
  bbox_inches='tight',
)

plt.show()
plt.close(fig)


## Continuum scan — diagonal and kernel-correlated

Each case is serialized immediately to its own range/mode folder, matching the reference scan organization and preventing all fit covariance objects from accumulating in memory.

In [ ]:
def snapshot_case(window, mode):
    payload = {
        'window': window, 'mode': mode, 'fit_id': FIT_ID, 'order': ORDER,
        'g2s': copy.deepcopy(bf.g2s), 'betas': copy.deepcopy(bf.betas),
        'cnt_fits': copy.deepcopy(bf.cnt_fits), 'cnt_qof': copy.deepcopy(bf.continuum.quality),
        'domains': copy.deepcopy(bf.continuum.domains),
        'metadata': copy.deepcopy(bf.continuum.metadata),
    }
    path = case_dir(window, mode) / 'continuum_case.gvar'
    with path.open('wb') as stream:
        gv.dump(payload, stream)

def load_case(window, mode):
    path = case_dir(window, mode) / 'continuum_case.gvar'
    with path.open('rb') as stream:
        return gv.load(stream)

mng2, mxg2, dg2 = G2_GRID
for window in WINDOWS:
    bf.cnt_xtrp(mnt=window[0], mxt=window[1], mng2=mng2, mxg2=mxg2, dg2=dg2,
                 cov_mode='diagonal', diagonal=True, correlated=False,
                 error_mode='fit', tau0=config.tau0, v=0)
    snapshot_case(window, 'diagonal')
    bf.cnt_xtrp(mnt=window[0], mxt=window[1], mng2=mng2, mxg2=mxg2, dg2=dg2,
                 cov_mode='kernel', diagonal=False, correlated=True,
                 error_mode='fit', tau0=config.tau0, v=0)
    snapshot_case(window, 'correlated')
    print('saved', window)
    gc.collect()

print(f'saved {2 * len(WINDOWS)} continuum cases')

## Plot 3 — intermediate interpolation for every flow-time range

In [ ]:
for window in WINDOWS:
     fig, ax = plt.subplots(figsize=(8, 6))

     for operator in OBSERVABLES:
         for time in available_flow_times(window):
             x, y, data = bf.interpolation_curve(FLOW, operator, time)
             ratio = np.asarray(y, dtype=object) / x**2
             alpha = 0.25 + .65 * (float(time) - window[0]) / max(
                 window[1] - window[0], 1e-12
             )

             ax.plot(
                 x,
                 gv.mean(ratio),
                 color=OP_COLORS[operator],
                 alpha=alpha,
                 lw=1.2,
             )
             ax.fill_between(
                 x,
                 gv.mean(ratio) - gv.sdev(ratio),
                 gv.mean(ratio) + gv.sdev(ratio),
                 color=OP_COLORS[operator],
                 alpha=.04,
             )

     xpt = np.linspace(.01, 5, 300)
     pt_curves = (
         (1, "-", "1-loop universal"),
         (2, "--", "2-loop universal"),
         (3, "-.", "3-loop gradient flow"),
     )
     for loops, ls, label in pt_curves:
         ax.plot(
             xpt,
             pt_over_g4(xpt, loops),
             color="gray",
             ls=ls,
             label=label,
         )

     # Preliminary-analysis watermark and analysis details.
     ax.text(
         0.50,
         0.52,
         r"\textbf{Preliminary}",
         transform=ax.transAxes,
         fontsize=42,
         color="gray",
         alpha=.28,
         ha="center",
         va="center",
         rotation=30,
         zorder=0,
     )
     ax.text(
         0.58,
         0.16,
         (
             FIT_WATERMARK
             + "\n"
             rf"Interpolation, $t/a^2\in[{window[0]:g},{window[1]:g}]$"
             "\n"
             r"TLN"
         ),
         transform=ax.transAxes,
         fontsize=10.5,
         color="gray",
         alpha=.52,
         ha="center",
         va="center",
         fontweight="bold",
     )

     op_handles = [
         Line2D(
             [0],
             [0],
             color=OP_COLORS[operator],
             label=OP_LABELS[operator],
         )
         for operator in OBSERVABLES
     ]
     handles, labels = ax.get_legend_handles_labels()
     ax.legend(
         op_handles + handles,
         [handle.get_label() for handle in op_handles] + labels,
         ncol=2,
         frameon=False,
     )

     ax.set(
         xlim=(0, 5),
         ylim=(-0.065, -0.038),
         xlabel=r"$g^2_{GF}$",
         ylabel=r"$\beta_{GF}/g_{GF}^4$",
       #  title=rf"Interpolation, $t/a^2\in[{window[0]:g},{window[1]:g}]$",
     )

     save_figure(
         fig,
         window,
         "interpolation_beta_over_g4_fit2",
     )


In [ ]:
# Plot 3b — interpolation curves together with the fitted IV data points

  # ------------------------------------------------------------------
  # Plot controls
# ------------------------------------------------------------------
PLOT_TIME_STEP = 0.25

# Choose which operators to plot:
PLOT_OPERATORS = ('p', 'c', 's')  # all operators
# PLOT_OPERATORS = ('p',)         # Wilson only
# PLOT_OPERATORS = ('c',)         # clover only
# PLOT_OPERATORS = ('s',)         # Symanzik only


def select_flow_times(available_times, window, step):
  """Select available flow times nearest to a regular grid with spacing step."""
  available_times = sorted(available_times, key=float)

  if not available_times:
      return []

  targets = np.arange(
      window[0],
      window[1] + 0.5 * step,
      step,
  )

  selected = []
  for target in targets:
      nearest = min(
          available_times,
          key=lambda time: abs(float(time) - target),
      )

      if nearest not in selected:
          selected.append(nearest)

  return selected


for window in WINDOWS:
  fig, ax = plt.subplots(figsize=(8, 6))
  coefficient_lines = []

  for operator in PLOT_OPERATORS:
      available_times = available_flow_times(window)
      times = select_flow_times(
          available_times,
          window,
          PLOT_TIME_STEP,
      )

      # Quote the coefficients for the strongest (latest-time) curve.
      if times:
          coefficient_time = times[-1]
          coefficient_fit = bf.interpolation.fetch(
              'fits', (FLOW, operator, coefficient_time),
          )
          coefficient_values = ', '.join(
              rf'$p_{{{index}}}={format(coefficient_fit[f"p{index}"][0], "2p")}$'
              for index in POLYNOMIAL_PARAMETER_INDICES
          )
          coefficient_lines.append(
              rf'{OP_LABELS[operator]}, $t/a^2={float(coefficient_time):g}$: '
              + coefficient_values
          )

      for time in times:
          x_grid, beta_grid, data = bf.interpolation_curve(
              FLOW,
              operator,
              time,
          )

          x_grid = np.asarray(x_grid, dtype=float)
          beta_grid = np.asarray(beta_grid, dtype=object)

          # x = g_GF^2, so g_GF^4 = x^2.
          curve_ratio = beta_grid / x_grid**2

          # Infinite-volume data points supplied to the interpolation fit.
          data_x = np.asarray(data.x, dtype=object)
          data_beta = np.asarray(data.y, dtype=object)
          data_ratio = data_beta / data_x**2

          # Later flow times are drawn more strongly.
          alpha = 0.25 + 0.65 * (
              (float(time) - window[0])
              / max(window[1] - window[0], 1e-12)
          )

          color = OP_COLORS[operator]

          # Interpolation central value.
          ax.plot(
              x_grid,
              gv.mean(curve_ratio),
              color=color,
              alpha=alpha,
              lw=1.5,
          )

          # One-sigma interpolation band.
          ax.fill_between(
              x_grid,
              gv.mean(curve_ratio) - gv.sdev(curve_ratio),
              gv.mean(curve_ratio) + gv.sdev(curve_ratio),
              color=color,
              alpha=0.08,
              linewidth=0,
          )

          # Infinite-volume data used in the fit.
          ax.errorbar(
              gv.mean(data_x),
              gv.mean(data_ratio),
              xerr=gv.sdev(data_x),
              yerr=gv.sdev(data_ratio),
              fmt="o",
              ms=4.5,
              capsize=2.5,
              elinewidth=0.9,
              color=color,
              alpha=alpha,
              markeredgecolor="black",
              markeredgewidth=0.35,
              linestyle="none",
              zorder=3,
          )

  # Perturbative reference curves.
  x_pt = np.linspace(0.01, 5.0, 300)

  pt_curves = (
      (1, "-", "1-loop universal"),
      (2, "--", "2-loop universal"),
      (3, "-.", "3-loop gradient flow"),
  )

  pt_handles = []

  for loops, linestyle, label in pt_curves:
      line, = ax.plot(
          x_pt,
          pt_over_g4(x_pt, loops),
          color="gray",
          linestyle=linestyle,
          lw=1.4,
          alpha=0.7,
          label=label,
      )
      pt_handles.append(line)

  # One legend entry for each selected operator.
  operator_handles = [
      Line2D(
          [0],
          [0],
          color=OP_COLORS[operator],
          lw=1.5,
          marker="o",
          markersize=5,
          markeredgecolor="black",
          markeredgewidth=0.35,
          label=OP_LABELS[operator],
      )
      for operator in PLOT_OPERATORS
  ]

  ax.legend(
      handles=operator_handles + pt_handles,
      ncol=2,
      loc='upper left',
      frameon=False,
  )

  ax.text(
      0.98, 0.98, '\n'.join(coefficient_lines),
      transform=ax.transAxes, ha='right', va='top', fontsize=7.5,
      bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                edgecolor='gray', alpha=0.88),
      zorder=10,
  )

  ax.set(
      xlim=(0, 5),
      ylim=(-0.065, -0.038),
      xlabel=r"$g^2_{GF}$",
      ylabel=r"$\beta_{GF}/g_{GF}^4$",
  )

  ax.minorticks_on()
  ax.grid(which="major", linestyle="-", alpha=0.35)
  ax.grid(which="minor", linestyle="--", alpha=0.20)

  ax.text(
      0.50,
      0.52,
      r"\textbf{Preliminary}",
      transform=ax.transAxes,
      fontsize=42,
      color="gray",
      alpha=0.28,
      ha="center",
      va="center",
      rotation=30,
      zorder=0,
  )

  ax.text(
      0.58,
      0.15,
      (
          FIT_WATERMARK
          + "\n"
          + rf"Interpolation with IV data, "
          + rf"$t/a^2\in[{window[0]:g},{window[1]:g}]$"
          + "\n"
          + rf"$\Delta(t/a^2)\simeq {PLOT_TIME_STEP:g}$"
          + "\n"
          + r"TLN"
      ),
      transform=ax.transAxes,
      fontsize=10.5,
      color="gray",
      alpha=0.52,
      ha="center",
      va="center",
      fontweight="bold",
  )

  fig.tight_layout()

  save_figure(
      fig,
      window,
      "interpolation_beta_over_g4_fit2_with_data",
  )


## Standardized interpolation residuals versus $g^2$

For every flow-time range, this plots $(\mathrm{fit}-\mathrm{data})/\sigma_{\mathrm{data}}$ against measured $g^2_{GF}$. Separate panels show the selected operators, while color identifies flow time.

In [ ]:
# Plot 3c — beta_GF interpolation residuals versus g_GF^2
#
# Fitted quantity:
#
#   beta_GF = beta_GF(g_GF^2)
#
# Standardized residual:
#
#   z = (beta_GF^fit - beta_GF^data) / sigma_total
#
# with the broader quadrature uncertainty
#
#   sigma_total^2
#       = sigma_beta,data^2
#       + sigma_beta,fit^2
#
# Plot encoding:
# - every available flow-time point is included;
# - bare lattice beta determines the base hue;
# - smallest flow time is darkest;
# - largest flow time is lighter;
# - no legend or marker borders;
# - horizontal bars show sigma_(g_GF^2);
# - vertical bars show the normalized total uncertainty, ±1.

from matplotlib.colors import to_rgb


def nested_key_exists(mapping, keys):
  """Return True if all keys exist along a nested-dictionary path."""
  target = mapping

  try:
      for key in keys:
          target = target[key]
  except (KeyError, TypeError):
      return False

  return True


def couplings_for_fit_input(operator, time_key):
  """
  Recover the coupling order used to construct the interpolation FitInput.

  A coupling contributes when both its g2 and beta IV fits exist for
  the selected flow, operator, and exact flow-time key.
  """
  couplings = []

  for coupling in bf.iv_fits:
      g2_path = (
          coupling,
          'g2',
          FLOW,
          operator,
          time_key,
      )
      beta_path = (
          coupling,
          'beta',
          FLOW,
          operator,
          time_key,
      )

      if (
          nested_key_exists(bf.iv_fits, g2_path)
          and nested_key_exists(bf.iv_fits, beta_path)
      ):
          couplings.append(coupling)

  return couplings


for window in WINDOWS:
  fig, axes = plt.subplots(
      1,
      len(PLOT_OPERATORS),
      figsize=(5.2 * len(PLOT_OPERATORS), 4.8),
      sharex=True,
      sharey=True,
      squeeze=False,
  )
  axes = axes.ravel()

  # Preserve the original StageStore flow-time keys.
  time_keys = sorted(
      available_flow_times(window),
      key=float,
  )

  if not time_keys:
      plt.close(fig)
      print(f'No flow times available for window {window}.')
      continue

  # Numeric versions are used only for shading and text.
  time_values = np.asarray(
      [float(time_key) for time_key in time_keys],
      dtype=float,
  )

  time_min = float(time_values.min())
  time_max = float(time_values.max())

  # Collect the bare lattice-beta values.
  all_beta_values = []

  for operator in PLOT_OPERATORS:
      for time_key in time_keys:
          couplings = couplings_for_fit_input(
              operator,
              time_key,
          )

          all_beta_values.extend(
              beta_value(coupling)
              for coupling in couplings
          )

  beta_values = np.asarray(
      sorted(set(all_beta_values)),
      dtype=float,
  )

  if beta_values.size == 0:
      plt.close(fig)
      print(
          f'No beta/coupling points found for window {window}.'
      )
      continue

  # Give every bare lattice beta a distinct base hue.
  beta_cmap = plt.get_cmap(
      'tab10' if len(beta_values) <= 10 else 'turbo'
  )

  if len(beta_values) == 1:
      beta_color_positions = np.asarray([0.5])
  else:
      beta_color_positions = np.linspace(
          0.05,
          0.95,
          len(beta_values),
      )

  beta_colors = {
      float(beta): beta_cmap(position)
      for beta, position in zip(
          beta_values,
          beta_color_positions,
      )
  }

  def flowtime_color(beta, time_value):
      """
      Keep the beta-dependent hue while varying its shade.

      Smallest flow time -> darkest.
      Largest flow time  -> lighter, but not pale.
      """
      base_rgb = np.asarray(
          to_rgb(beta_colors[float(beta)]),
          dtype=float,
      )

      if time_max == time_min:
          time_fraction = 0.0
      else:
          time_fraction = (
              (float(time_value) - time_min)
              / (time_max - time_min)
          )

      dark_rgb = 0.50 * base_rgb

      light_rgb = (
          0.82 * base_rgb
          + 0.18 * np.ones(3)
      )

      return tuple(
          (1.0 - time_fraction) * dark_rgb
          + time_fraction * light_rgb
      )

  for ax, operator in zip(
      axes,
      PLOT_OPERATORS,
  ):
      for time_key in time_keys:
          # Use the exact key for StageStore access.
          time_value = float(time_key)

          data = bf.interpolation.fetch(
              'inputs',
              (FLOW, operator, time_key),
          )
          params = bf.interpolation.fetch(
              'fits',
              (FLOW, operator, time_key),
          )

          # Interpolation inputs:
          #   x = g_GF^2
          #   y = beta_GF
          data_x = np.asarray(
              data.x,
              dtype=object,
          )
          data_beta_gf = np.asarray(
              data.y,
              dtype=object,
          )

          # Recover the bare lattice beta corresponding to each point.
          couplings = couplings_for_fit_input(
              operator,
              time_key,
          )

          point_betas = np.asarray(
              [
                  beta_value(coupling)
                  for coupling in couplings
              ],
              dtype=float,
          )

          if len(point_betas) != len(data_x):
              raise RuntimeError(
                  'Could not match interpolation points to bare-beta '
                  f'values for operator={operator}, '
                  f'time={time_key}: '
                  f'{len(data_x)} data points but '
                  f'{len(point_betas)} coupling labels.'
              )

          # When x errors are fitted, params['x'] contains the
          # corresponding posterior x values.
          fitted_x = (
              np.asarray(
                  params['x'],
                  dtype=object,
              )
              if 'x' in params
              else data_x
          )

          # Prediction of the directly fitted quantity beta_GF.
          fitted_beta_gf = np.asarray(
              [
                  bf.interpolation.model.evaluate(
                      value,
                      params,
                  )
                  for value in fitted_x
              ],
              dtype=object,
          )

          # Central beta-function residual.
          residual_mean = np.asarray(
              gv.mean(fitted_beta_gf)
              - gv.mean(data_beta_gf),
              dtype=float,
          )

          # Uncertainty of the input beta_GF data.
          sigma_beta_data = np.asarray(
              gv.sdev(data_beta_gf),
              dtype=float,
          )

          # Uncertainty of the fitted beta_GF prediction.
          sigma_beta_fit = np.asarray(
              gv.sdev(fitted_beta_gf),
              dtype=float,
          )

          # Broader quadrature total uncertainty.
          #
          # This does not apply the potentially strong in-sample
          # covariance cancellation from sdev(fit - data).
          sigma_beta_total = np.sqrt(
              sigma_beta_data**2
              + sigma_beta_fit**2
          )

          standardized_residual = np.divide(
              residual_mean,
              sigma_beta_total,
              out=np.full(
                  len(data_beta_gf),
                  np.nan,
                  dtype=float,
              ),
              where=(
                  np.isfinite(sigma_beta_total)
                  & (sigma_beta_total > 0)
              ),
          )

          # Horizontal coordinate and its uncertainty.
          x_mean = np.asarray(
              gv.mean(data_x),
              dtype=float,
          )
          sigma_x = np.asarray(
              gv.sdev(data_x),
              dtype=float,
          )

          valid = (
              np.isfinite(x_mean)
              & np.isfinite(sigma_x)
              & np.isfinite(standardized_residual)
              & np.isfinite(sigma_beta_total)
              & np.isfinite(point_betas)
          )

          for beta in beta_values:
              selected = (
                  valid
                  & np.isclose(
                      point_betas,
                      beta,
                      rtol=0.0,
                      atol=1e-12,
                  )
              )

              if not np.any(selected):
                  continue

              point_color = flowtime_color(
                  beta,
                  time_value,
              )

              # Since the vertical quantity is divided by sigma_total,
              # the corresponding normalized vertical uncertainty is 1.
              normalized_yerr = np.ones(
                  np.count_nonzero(selected),
                  dtype=float,
              )

              ax.errorbar(
                  x_mean[selected],
                  standardized_residual[selected],
                  xerr=sigma_x[selected],
                  yerr=normalized_yerr,
                  fmt='o',
                  markersize=5.5,
                  color=point_color,
                  ecolor=point_color,
                  elinewidth=0.8,
                  capsize=2,
                  alpha=0.95,
                  markeredgecolor='none',
                  markeredgewidth=0,
                  linestyle='none',
                  zorder=3,
              )

      ax.axhline(
          0,
          color='black',
          lw=1.0,
      )

      ax.axhspan(
          -1,
          1,
          color='gray',
          alpha=0.12,
      )

      for level, style in (
          (1, '--'),
          (-1, '--'),
          (2, ':'),
          (-2, ':'),
      ):
          ax.axhline(
              level,
              color='gray',
              lw=0.8,
              ls=style,
          )

      ax.set_title(
          OP_LABELS[operator]
      )
      ax.set_xlabel(
          r'$g^2_{GF}$'
      )
      ax.grid(
          alpha=0.22
      )

  axes[0].set_ylabel(
      r'$(\beta_{GF}^{\mathrm{fit}}'
      r'-\beta_{GF}^{\mathrm{data}})/\sigma$'
  )

  fig.suptitle(
      FIT_WATERMARK
      + '\n'
      + (
          rf'Interpolation residuals for $\beta_{{GF}}$, '
          rf'$t/a^2\in[{time_min:g},{time_max:g}]$'
      )
      + '\n'
      + (
          r'quadrature total uncertainty; '
          r'all available flow times; '
          r'smallest flow time darkest, '
          r'largest lighter; TLN'
      ),
      fontsize=11,
      color='gray',
      fontweight='bold',
  )

  fig.subplots_adjust(
      top=0.76,
      right=0.97,
      wspace=0.12,
  )

  save_figure(
      fig,
      window,
      (
          'interpolation_beta_GF_residuals_'
          'over_sigma_total_'
          'all_flowtimes_fit2'
      ),
  )

## Detailed interpolation diagnostics

This reports lsqfit chi-squared, degrees of freedom, chi-squared per degree of freedom, p-value, logGBF, fitted coefficients, and each coupling's signed marginal residual in standard deviations. The Fit4 coefficients have no priors and coupling uncertainties are not promoted to latent fit variables (`xerrors=False`), matching the no-prior runner configuration.


In [ ]:
interpolation_detail_rows = []

for window in WINDOWS:
    for operator in OBSERVABLES:
        times = [time for time in sorted(bf.ntrp_fits[FLOW][operator], key=float)
                 if window[0] <= float(time) <= window[1]]
        for time in times:
            data = bf.interpolation.fetch('inputs', (FLOW, operator, time))
            params = bf.interpolation.fetch('fits', (FLOW, operator, time))
            qof = bf.interpolation.fetch('quality', (FLOW, operator, time))
            data_x = np.asarray(data.x, dtype=object)
            data_y = np.asarray(data.y, dtype=object)
            fitted_x = np.asarray(params['x'], dtype=object) if 'x' in params else data_x
            prediction = np.asarray([bf.interpolation.model.evaluate(value, params)
                                     for value in fitted_x], dtype=object)
            residual = data_y - prediction
            residual_sdev = np.asarray(gv.sdev(residual), dtype=float)
            pull = np.divide(gv.mean(residual), residual_sdev,
                             out=np.full(len(residual), np.nan),
                             where=residual_sdev > 0)
            fit_couplings = [coupling for coupling in IV_COUPLINGS
                             if time in bf.iv_fits[coupling]['g2'][FLOW][operator]
                             and time in bf.iv_fits[coupling]['beta'][FLOW][operator]]
            if len(fit_couplings) != len(data_y):
                raise RuntimeError(f'Coupling/data mismatch for {operator}, t={time}: '
                                   f'{len(fit_couplings)} versus {len(data_y)}')
            chi2 = float(qof['chi2'])
            dof = int(qof['dof'])
            parameter_summary = '; '.join(f'{name}={params[name][0]}'
                                          for name in sorted(params) if name != 'x')
            for index, coupling in enumerate(fit_couplings):
                interpolation_detail_rows.append({
                    'prior_width': FIT_WIDTH, 'window': window_tag(window),
                    'operator': operator, 'operator_label': OP_LABELS[operator],
                    'flow_time': float(time), 'coupling': coupling,
                    'beta_b': beta_value(coupling),
                    'g2_mean': float(gv.mean(data_x[index])),
                    'g2_sdev': float(gv.sdev(data_x[index])),
                    'fitted_g2_mean': float(gv.mean(fitted_x[index])),
                    'beta_mean': float(gv.mean(data_y[index])),
                    'beta_sdev': float(gv.sdev(data_y[index])),
                    'prediction_mean': float(gv.mean(prediction[index])),
                    'prediction_sdev': float(gv.sdev(prediction[index])),
                    'residual_mean': float(gv.mean(residual[index])),
                    'residual_sdev': float(residual_sdev[index]),
                    'signed_sigma_deviation': float(pull[index]),
                    'abs_sigma_deviation': float(abs(pull[index])),
                    'chi2_including_priors': chi2, 'dof_reported': dof,
                    'chi2_per_dof': chi2 / dof if dof else np.nan,
                    'p_value': float(qof['p-value']), 'logGBF': qof.get('logGBF'),
                    'n_data_points': len(data_y), 'n_coefficient_priors': FIT_PRIOR_COUNT,
                    'xerrors': config.interpolation.xerrors,
                    'posterior_parameters': parameter_summary,
                })

interpolation_diagnostics = pd.DataFrame(interpolation_detail_rows)
fit_columns = ['prior_width', 'window', 'operator', 'operator_label', 'flow_time',
               'chi2_including_priors', 'dof_reported', 'chi2_per_dof',
               'p_value', 'logGBF', 'n_data_points', 'n_coefficient_priors',
               'xerrors', 'posterior_parameters']
interpolation_fit_summary = (interpolation_diagnostics[fit_columns]
    .drop_duplicates(['window', 'operator', 'flow_time'])
    .sort_values(['window', 'operator', 'flow_time']).reset_index(drop=True))
interpolation_diagnostics.to_csv(OUTPUT_ROOT / 'interpolation_diagnostics_by_coupling.csv', index=False)
interpolation_fit_summary.to_csv(OUTPUT_ROOT / 'interpolation_fit_quality_summary.csv', index=False)
(OUTPUT_ROOT / 'interpolation_fit_quality_summary.txt').write_text(
    interpolation_fit_summary.to_string(index=False) + '\n'
)
for window in WINDOWS:
    tag = window_tag(window)
    interpolation_diagnostics.query('window == @tag').to_csv(
        case_dir(window) / 'interpolation_diagnostics_by_coupling.csv', index=False)
    interpolation_fit_summary.query('window == @tag').to_csv(
        case_dir(window) / 'interpolation_fit_quality_summary.csv', index=False)
    (case_dir(window) / 'interpolation_fit_quality_summary.txt').write_text(
        interpolation_fit_summary.query('window == @tag').to_string(index=False) + '\n'
    )
display(interpolation_fit_summary)
display(interpolation_diagnostics.sort_values('abs_sigma_deviation', ascending=False).head(30))


## Plot 4 — interpolation quality for every flow-time range

In [ ]:
for window in WINDOWS:
      fig, (ax1, ax2) = plt.subplots(
          2,
          1,
          figsize=(8, 8),
          sharex=True,
      )

      for operator in OBSERVABLES:
          rows = []
          for time, qof in bf.ntrp_qof[FLOW][operator].items():
              if window[0] <= float(time) <= window[1]:
                  rows.append(
                      (
                          float(time),
                          qof['chi2'] / qof['dof']
                          if qof['dof']
                          else np.nan,
                          qof['p-value'],
                      )
                  )

          if rows:
              rows = np.asarray(sorted(rows), dtype=float)
              ax1.plot(
                  rows[:, 0],
                  rows[:, 1],
                  'o-',
                  color=OP_COLORS[operator],
                  label=OP_LABELS[operator],
              )
              ax2.plot(
                  rows[:, 0],
                  rows[:, 2],
                  'o-',
                  color=OP_COLORS[operator],
              )

      # Draw watermark above all plot elements.
      ax1.text(
          0.50,
          0.52,
          r'\textbf{Preliminary}',
          transform=ax1.transAxes,
          fontsize=42,
          color='gray',
          alpha=.28,
          ha='center',
          va='center',
          rotation=30,
          zorder=20,
          clip_on=False,
      )
      ax1.text(
          0.58,
          0.16,
          (
              FIT_WATERMARK
              + '\n'
              rf'Interpolation QoF, $t/a^2\in[{window[0]:g},{window[1]:g}]$'
              '\n'
              r'TLN'
          ),
          transform=ax1.transAxes,
          fontsize=10.5,
          color='gray',
          alpha=.52,
          ha='center',
          va='center',
          fontweight='bold',
          zorder=20,
          clip_on=False,
      )

      ax1.axhline(1, color='gray', ls='--')
      ax2.axhline(.05, color='gray', ls='--')

      ax1.set_ylabel(r'$\chi^2/{\rm dof}$')
      ax2.set_ylabel(r'$p$-value')
      ax2.set_xlabel(r'$t/a^2$')

      ax1.legend(frameon=False)

      save_figure(
          fig,
          window,
          'interpolation_quality_fit2',
      )

## Plot helper — continuum extrapolations versus $a^2/t$

In [ ]:
def continuum_panel_times(case, window, operator, g2, tau0):
      """Return fit times and one terminal point one flow-time unit beyond each edge."""
      all_times = sorted(bf.ntrp_fits[FLOW][operator], key=float)
      valid_times = [
          time for time in all_times
          if float(time) - tau0 > 0.0
          and bf.ntrp_nf[FLOW][operator][time][0]
          <= g2
          <= bf.ntrp_nf[FLOW][operator][time][-1]
      ]

      domain_by_g2 = (
          case.get('domains', {}).get(FLOW, {}).get(operator, {})
      )
      fit_times = []
      if domain_by_g2:
          domain_key = min(
              domain_by_g2, key=lambda key: abs(float(key) - g2)
          )
          if np.isclose(float(domain_key), g2, rtol=0.0, atol=1e-12):
              fit_times = list(domain_by_g2[domain_key])
      if not fit_times:
          fit_times = [
              time for time in valid_times
              if window[0] <= float(time) - tau0 <= window[1]
          ]
      fit_times = sorted(fit_times, key=float)

      valid_nominal = np.asarray(
          [float(time) - tau0 for time in valid_times], dtype=float
      )
      valid_steps = np.diff(valid_nominal)
      valid_steps = valid_steps[valid_steps > 1e-10]
      grid_tolerance = (
          0.51 * float(np.median(valid_steps))
          if len(valid_steps)
          else 1e-8
      )

      terminal_targets = (window[0] - 1.0, window[1] + 1.0)
      outside_times = []
      for target in terminal_targets:
          candidates = [
              time for time in valid_times
              if window[0] - 1.0 - 1e-10
              <= float(time) - tau0
              <= window[1] + 1.0 + 1e-10
              and not window[0]
              <= float(time) - tau0
              <= window[1]
          ]
          if not candidates:
              continue
          nearest = min(
              candidates,
              key=lambda time: abs((float(time) - tau0) - target),
          )
          if (
              abs((float(nearest) - tau0) - target)
              <= grid_tolerance
              and nearest not in outside_times
          ):
              outside_times.append(nearest)
      outside_times = sorted(outside_times, key=float)
      return fit_times, outside_times


def plot_continuum_panels(window, mode, divide_by_g4):
      case = load_case(window, mode)
      fig, axes = plt.subplots(2, 4, figsize=(18, 9))

      tau0 = config.tau0
      correction_label = (
          'TLN'
          if config.correction in ('tln', 'tree-level-normalization')
          else 'No TLN'
      )
      mode_label = (
          'Correlated continuum'
          if mode == 'correlated'
          else 'Diagonal continuum'
      )

      for ax, target in zip(axes.ravel(), TARGET_G2):
          displayed_g2 = []

          for operator in OBSERVABLES:
              grid = np.asarray(
                  case['g2s'][FLOW][operator],
                  dtype=float,
              )

              if len(grid) == 0:
                  continue

              # Use the continuum point nearest to the requested target.
              idx = int(np.argmin(np.abs(grid - target)))
              g2 = float(grid[idx])
              displayed_g2.append(g2)

              params = case['cnt_fits'][FLOW][operator][idx]

              fit_times, outside_times = continuum_panel_times(
                  case, window, operator, g2, tau0
              )

              def continuum_plot_values(times):
                  nominal_times = np.asarray(
                      [float(time) - tau0 for time in times],
                      dtype=float,
                  )
                  measured_times = np.asarray(
                      [float(time) for time in times],
                      dtype=float,
                  )

                  xvalues = 1.0 / nominal_times
                  jacobian = nominal_times / measured_times

                  yvalues = np.asarray(
                      [
                          factor
                          * bf.interpolation.model.evaluate(
                              g2,
                              bf.ntrp_fits[FLOW][operator][time],
                          )
                          for factor, time in zip(jacobian, times)
                      ],
                      dtype=object,
                  )

                  norm = g2**2 if divide_by_g4 else 1.0
                  return xvalues, yvalues / norm

              norm = g2**2 if divide_by_g4 else 1.0

              # Filled points: data included in the continuum fit.
              if fit_times:
                  xfit_data, yfit_data = continuum_plot_values(fit_times)

                  ax.errorbar(
                      xfit_data,
                      gv.mean(yfit_data),
                      yerr=gv.sdev(yfit_data),
                      fmt='o',
                      ms=5,
                      capsize=3,
                      color=OP_COLORS[operator],
                      markerfacecolor=OP_COLORS[operator],
                      markeredgecolor=OP_COLORS[operator],
                      label=OP_LABELS[operator],
                      zorder=4,
                  )

              # Hollow points: one terminal point one flow-time unit beyond each edge.
              if outside_times:
                  xoutside, youtside = continuum_plot_values(outside_times)

                  ax.errorbar(
                      xoutside,
                      gv.mean(youtside),
                      yerr=gv.sdev(youtside),
                      fmt='o',
                      ms=5,
                      capsize=3,
                      color=OP_COLORS[operator],
                      markerfacecolor='none',
                      markeredgecolor=OP_COLORS[operator],
                      zorder=4,
                  )

              # Draw the fitted extrapolation over the displayed a²/t range.
              displayed_times = fit_times + outside_times
              if displayed_times:
                  displayed_x = np.asarray(
                      [
                          1.0 / (float(time) - tau0)
                          for time in displayed_times
                      ],
                      dtype=float,
                  )
                  xmax = 1.05 * np.max(displayed_x)
              else:
                  xmax = 1.0 / max(window[0] - tau0, 1e-12)

              xline = np.linspace(0.0, xmax, 200)
              yline = (
                  params['beta'][0]
                  + params['slope'][0] * xline
              ) / norm

              ax.plot(
                  xline,
                  gv.mean(yline),
                  color=OP_COLORS[operator],
                  lw=1.5,
                  zorder=2,
              )
              ax.fill_between(
                  xline,
                  gv.mean(yline) - gv.sdev(yline),
                  gv.mean(yline) + gv.sdev(yline),
                  color=OP_COLORS[operator],
                  alpha=.15,
                  zorder=1,
              )

          if displayed_g2:
              panel_g2 = displayed_g2[0]
              ax.text(
                  .04,
                  .94,
                  rf'$g^2_{{GF}}={panel_g2:g}$',
                  transform=ax.transAxes,
                  ha='left',
                  va='top',
                  fontsize=13,
                  zorder=20,
              )

          ax.set_xlabel(r'$a^2/t$')

      axes.ravel()[0].legend(frameon=False)

      ylabel = (
          r'$\beta_{GF}/g_{GF}^4$'
          if divide_by_g4
          else r'$\beta_{GF}$'
      )
      fig.supylabel(ylabel)

      # Figure-level watermark and analysis information.
      fig.text(
          .50,
          .52,
          r'\textbf{Preliminary}',
          fontsize=58,
          color='gray',
          alpha=.18,
          ha='center',
          va='center',
          rotation=30,
          zorder=100,
      )
      fig.text(
          .50,
          .025,
          (
              FIT_WATERMARK
              + r'; '
              + mode_label
              + r'; '
              + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$'
              + r'; '
              + correction_label
              + '\n'
              + 'Filled points: included in fit; '
              + 'hollow points: terminal values one flow-time unit beyond each boundary'
          ),
          fontsize=11,
          color='gray',
          alpha=.9,
          ha='center',
          va='bottom',
          fontweight='bold',
          zorder=100,
      )

      fig.tight_layout(rect=(.025, .075, 1, 1))

      name = (
          'continuum_a2_over_t_beta_over_g4_fit2'
          if divide_by_g4
          else 'continuum_a2_over_t_beta_fit2'
      )
      save_figure(fig, window, name, mode)

## Plot 5 — raw $eta_{GF}$ continuum panels for all ranges and modes

In [ ]:
for window in WINDOWS:
    for mode in ('diagonal', 'correlated'):
        plot_continuum_panels(window, mode, divide_by_g4=False)

## Plot 6 — $eta_{GF}/g_{GF}^4$ continuum panels for all ranges and modes

In [ ]:
for window in WINDOWS:
    for mode in ('diagonal', 'correlated'):
        plot_continuum_panels(window, mode, divide_by_g4=True)

## Plot 7 — continuum fit quality for all ranges and modes

In [ ]:
for window in WINDOWS:
      for mode in ('diagonal', 'correlated'):
          case = load_case(window, mode)
          fig, (ax1, ax2) = plt.subplots(
              2,
              1,
              figsize=(8, 8),
              sharex=True,
          )

          for operator in OBSERVABLES:
              grid = np.asarray(
                  case['g2s'][FLOW][operator],
                  dtype=float,
              )
              q = case['cnt_qof'][FLOW][operator]

              chi = np.asarray(
                  [
                      row['chi2'] / row['dof']
                      if row['dof']
                      else np.nan
                      for row in q
                  ],
                  dtype=float,
              )
              pv = np.asarray(
                  [row['p-value'] for row in q],
                  dtype=float,
              )

              # Ensure that every available continuum-fit point is plotted.
              npoints = min(len(grid), len(chi), len(pv))
              grid_plot = grid[:npoints]
              chi_plot = chi[:npoints]
              pv_plot = pv[:npoints]

              ax1.plot(
                  grid_plot,
                  chi_plot,
                  '-',
                  color=OP_COLORS[operator],
                  lw=1.2,
                  alpha=.8,
              )
              ax1.scatter(
                  grid_plot,
                  chi_plot,
                  color=OP_COLORS[operator],
                  s=28,
                  zorder=3,
                  label=OP_LABELS[operator],
              )

              ax2.plot(
                  grid_plot,
                  pv_plot,
                  '-',
                  color=OP_COLORS[operator],
                  lw=1.2,
                  alpha=.8,
              )
              ax2.scatter(
                  grid_plot,
                  pv_plot,
                  color=OP_COLORS[operator],
                  s=28,
                  zorder=3,
              )

          ax1.axhline(1, color='gray', ls='--')
          ax2.axhline(.05, color='gray', ls='--')

          ax1.set_ylabel(r'$\chi^2/{\rm dof}$')
          ax2.set_ylabel(r'$p$-value')
          ax2.set_xlabel(r'$g^2_{GF}$')
          ax1.legend(frameon=False)

          correction_label = (
              'TLN'
              if config.correction in ('tln', 'tree-level-normalization')
              else 'No TLN'
          )
          mode_label = (
              'Correlated continuum QoF'
              if mode == 'correlated'
              else 'Diagonal continuum QoF'
          )

          # Watermark.
          for ax in (ax1, ax2):
              ax.text(
                  .50,
                  .52,
                  r'\textbf{Preliminary}',
                  transform=ax.transAxes,
                  fontsize=36,
                  color='gray',
                  alpha=.25,
                  ha='center',
                  va='center',
                  rotation=30,
                  zorder=0,
              )
              ax.text(
                  .58,
                  .15,
                  (
                      FIT_WATERMARK
                      + '\n'
                      + mode_label
                      + '\n'
                      + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$'
                      + '\n'
                      + correction_label
                  ),
                  transform=ax.transAxes,
                  fontsize=10.5,
                  color='gray',
                  alpha=.92,
                  ha='center',
                  va='center',
                  fontweight='bold',
              )

          save_figure(
              fig,
              window,
              'continuum_quality_fit2',
              mode,
          )


## Plot helper — final continuum curves

In [ ]:
def plot_final_curve(window, mode, divide_by_g4):
      case = load_case(window, mode)
      fig, ax = plt.subplots(figsize=(8, 6))

      correction_label = (
          'TLN'
          if config.correction in ('tln', 'tree-level-normalization')
          else 'No TLN'
      )
      mode_label = (
          'Correlated continuum'
          if mode == 'correlated'
          else 'Diagonal continuum'
      )
      quantity_label = (
          r'$\beta_{GF}/g_{GF}^4$'
          if divide_by_g4
          else r'$\beta_{GF}$'
      )

      # Continuum central curves and uncertainty bands.
      for operator in OBSERVABLES:
          x = np.asarray(
              case['g2s'][FLOW][operator],
              dtype=float,
          )
          y = np.asarray(
              case['betas'][FLOW][operator],
              dtype=object,
          )

          if divide_by_g4:
              # Keep the original scaled-plot range and y scaling.
              visible = (0.0 <= x) & (x <= 5.0)
              x = x[visible]
              y = y[visible]
              y = y / x**2

          # Sort explicitly so the lines and bands cannot cross because of
          # an unordered coupling grid.
          order = np.argsort(x)
          x = x[order]
          y = y[order]

          ymean = gv.mean(y)
          ysdev = gv.sdev(y)

          ax.plot(
              x,
              ymean,
              color=OP_COLORS[operator],
              lw=1.8,
              label=OP_LABELS[operator],
              zorder=3,
          )
          ax.fill_between(
              x,
              ymean - ysdev,
              ymean + ysdev,
              color=OP_COLORS[operator],
              alpha=.18,
              linewidth=0,
              zorder=2,
          )

      # Perturbative comparison curves over the complete displayed range.
      plot_xmax = (
          5.0 if divide_by_g4
          else max(G2_GRID[1], G2_T0) + .25
      )
      xp = np.linspace(.001, plot_xmax, 900)
      pt_curves = (
          (1, '-', '1-loop universal'),
          (2, '--', '2-loop universal'),
          (3, '-.', '3-loop gradient flow'),
      )

      for loops, linestyle, label in pt_curves:
          perturbative_ratio = pt_over_g4(xp, loops)

          # pt_over_g4 returns beta_PT/g^4. Restore beta_PT for the
          # unscaled beta plot.
          perturbative_curve = (
              perturbative_ratio
              if divide_by_g4
              else xp**2 * perturbative_ratio
          )

          ax.plot(
              xp,
              perturbative_curve,
              color='gray',
              ls=linestyle,
              lw=1.5,
              label=label,
              zorder=1,
          )

      # Watermark drawn inside the axes and above the plotted curves.
      ax.text(
          .50,
          .52,
          r'\textbf{Preliminary}',
          transform=ax.transAxes,
          fontsize=42,
          color='gray',
          alpha=.25,
          ha='center',
          va='center',
          rotation=30,
          zorder=20,
          clip_on=True,
      )
      ax.text(
          .58,
          .15,
          (
              FIT_WATERMARK
              + '\n'
              + mode_label
              + '\n'
              + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$'
              + '\n'
              + correction_label
              + '\n'
              + quantity_label
          ),
          transform=ax.transAxes,
          fontsize=10.5,
          color='gray',
          alpha=.82,
          ha='center',
          va='center',
          fontweight='bold',
          zorder=20,
          clip_on=True,
      )

      if not divide_by_g4:
          ax.axvline(
              G2_T0, color='black', linestyle=':', lw=1.5,
              label=rf'$g^2_{{\mathrm{{GF}}}}(t_0)=\frac{{8}}{{5}}\pi^2={G2_T0:.9f}$',
              zorder=1,
          )
          ax.set_ylim(-13.0, 0.5)

      ax.set(
          xlim=(0, plot_xmax),
          xlabel=r'$g^2_{GF}$',
          ylabel=quantity_label,
      )
      ax.legend(
          ncol=2,
          frameon=False,
      )

      name = (
          'continuum_beta_over_g4_vs_g2_fit2'
          if divide_by_g4
          else 'continuum_beta_vs_g2_fit2'
      )
      save_figure(
          fig,
          window,
          name,
          mode,
      )


## Plot 8 — final raw continuum curves for all ranges and modes

In [ ]:
for window in WINDOWS:
      for mode in ('diagonal', 'correlated'):
          plot_final_curve(
              window,
              mode,
              divide_by_g4=False,
          )


## Plot 9 — final scaled continuum curves for all ranges and modes

In [ ]:
for window in WINDOWS:
      for mode in ('diagonal', 'correlated'):
          plot_final_curve(
              window,
              mode,
              divide_by_g4=True,
          )


## Plot 10 — diagonal versus correlated continuum for every range

In [ ]:
for window in WINDOWS:
      diag = load_case(window, 'diagonal')
      corr = load_case(window, 'correlated')

      fig, ax = plt.subplots(figsize=(8, 6))

      correction_label = (
          'TLN'
          if config.correction in ('tln', 'tree-level-normalization')
          else 'No TLN'
      )

      for operator in OBSERVABLES:
          for case, linestyle, mode_label in (
              (diag, '--', 'diagonal'),
              (corr, '-', 'correlated'),
          ):
              x = np.asarray(
                  case['g2s'][FLOW][operator],
                  dtype=float,
              )
              y = (
                  np.asarray(
                      case['betas'][FLOW][operator],
                      dtype=object,
                  )
                  / x**2
              )

              # Preserve the original displayed range when determining
              # automatic y limits.
              visible = (0.0 <= x) & (x <= 4.8)
              x = x[visible]
              y = y[visible]

              # Sort the continuum grid before drawing the curve and band.
              order = np.argsort(x)
              x = x[order]
              y = y[order]

              ymean = gv.mean(y)
              ysdev = gv.sdev(y)

              # Dashed: diagonal continuum.
              # Solid: correlated continuum.
              ax.plot(
                  x,
                  ymean,
                  color=OP_COLORS[operator],
                  ls=linestyle,
                  lw=1.8,
                  label=f'{OP_LABELS[operator]} {mode_label}',
                  zorder=4 if mode_label == 'correlated' else 3,
              )

              # Uncertainty band associated with the corresponding curve.
              ax.fill_between(
                  x,
                  ymean - ysdev,
                  ymean + ysdev,
                  color=OP_COLORS[operator],
                  alpha=.18 if mode_label == 'correlated' else .07,
                  linewidth=0,
                  zorder=2 if mode_label == 'correlated' else 1,
              )

      # Perturbative beta/g^4 comparison curves.
      xp = np.linspace(.001, 5.0, 500)
      pt_curves = (
          (1, '-', '1-loop universal'),
          (2, '--', '2-loop universal'),
          (3, '-.', '3-loop gradient flow'),
      )

      for loops, linestyle, label in pt_curves:
          ax.plot(
              xp,
              pt_over_g4(xp, loops),
              color='gray',
              ls=linestyle,
              lw=1.5,
              label=label,
              zorder=1,
          )

      # Watermark and analysis information inside the plotting area.
      ax.text(
          .50,
          .52,
          r'\textbf{Preliminary}',
          transform=ax.transAxes,
          fontsize=42,
          color='gray',
          alpha=.25,
          ha='center',
          va='center',
          rotation=30,
          zorder=20,
          clip_on=True,
      )
      ax.text(
          .58,
          .15,
          (
              FIT_WATERMARK
              + '\n'
              + 'Diagonal vs correlated continuum'
              + '\n'
              + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$'
              + '\n'
              + correction_label
              + '\n'
              + r'Solid: correlated; dashed: diagonal'
          ),
          transform=ax.transAxes,
          fontsize=10.5,
          color='gray',
          alpha=.82,
          ha='center',
          va='center',
          fontweight='bold',
          zorder=20,
          clip_on=True,
      )

      ax.set(
          xlim=(0, 4.8),
          ylim=(-0.065, -0.038),
          xlabel=r'$g^2_{GF}$',
          ylabel=r'$\beta_{GF}/g_{GF}^4$',
      )
      ax.legend(
          ncol=2,
          frameon=False,
      )

      save_figure(
          fig,
          window,
          'diagonal_vs_correlated_fit2',
      )


## Plot 11 — flow-time correlation matrices for every range

## Plot 12 — correlated weak-coupling extrapolation for every range

Point error bars are continuum-point uncertainties; colored bands are posterior uncertainties of the shared fitted curve and are not expected to coincide.

In [ ]:
def ratio_model(x, p):
  u = np.asarray(x) / bf.perturbative_beta_function.nrm
  correction = 1.0
  term = 1.0

  for n in range(1, ORDER + 1):
      term *= u
      correction += p[f'c{n}'][0] * term

  return pt_over_g4(x, 3) * correction


for window in WINDOWS:
  case = load_case(window, 'correlated')
  fig, ax = plt.subplots(figsize=(8, 6))

  correction_label = (
      'TLN'
      if config.correction in ('tln', 'tree-level-normalization')
      else 'No TLN'
  )

  fits = {}
  minimum_couplings = []

  for operator in OBSERVABLES:
      x = np.asarray(
          case['g2s'][FLOW][operator],
          dtype=float,
      )
      y = (
          np.asarray(
              case['betas'][FLOW][operator],
              dtype=object,
          )
          / x**2
      )

      if len(x) == 0:
          continue

      # Sort the correlated continuum results.
      order = np.argsort(x)
      x = x[order]
      y = y[order]
      minimum_couplings.append(float(x[0]))

      continuum_mean = gv.mean(y)
      continuum_sdev = gv.sdev(y)

      prior = {
          f'c{n}': [gv.gvar(0, 10)]
          for n in range(1, ORDER + 1)
      }

      # The gvars in y retain the covariance of the correlated continuum
      # calculation. lsqfit therefore uses their full covariance matrix.
      fit = lsqfit.nonlinear_fit(
          data=(x, y),
          fcn=ratio_model,
          prior=prior,
      )
      fits[operator] = fit

      # Evaluate the fitted curve and its posterior uncertainty from
      # g²=0 through the complete correlated-continuum data range.
      xp = np.linspace(0.0, float(x[-1]), 600)
      yp = np.asarray(
          ratio_model(xp, fit.p),
          dtype=object,
      )

      fit_mean = gv.mean(yp)
      fit_sdev = gv.sdev(yp)

      # Original correlated-continuum uncertainty envelope. This is shown
      # only over the region where continuum results actually exist.
      ax.plot(
          x,
          continuum_mean,
          color=OP_COLORS[operator],
          ls=':',
          lw=1.4,
          alpha=.9,
          label=(
              f'{OP_LABELS[operator]} correlated continuum '
              r'($\pm1\sigma$)'
          ),
          zorder=4,
      )
      ax.fill_between(
          x,
          continuum_mean - continuum_sdev,
          continuum_mean + continuum_sdev,
          color=OP_COLORS[operator],
          alpha=.10,
          linewidth=0,
          zorder=2,
      )

      # PT-constrained weak-coupling fit and posterior uncertainty,
      # including the extrapolated interval down to g²=0.
      ax.plot(
          xp,
          fit_mean,
          color=OP_COLORS[operator],
          ls='-',
          lw=2.0,
          label=(
              f'{OP_LABELS[operator]} correlated '
              'weak-coupling fit'
          ),
          zorder=5,
      )
      ax.fill_between(
          xp,
          fit_mean - fit_sdev,
          fit_mean + fit_sdev,
          color=OP_COLORS[operator],
          alpha=.22,
          linewidth=0,
          zorder=3,
      )

      origin = np.asarray(
          ratio_model(np.asarray([0.0]), fit.p),
          dtype=object,
      )[0]

      print(
          window,
          OP_LABELS[operator],
          'correlated',
          'origin=',
          origin,
          'Q=',
          fit.Q,
      )

  # The shaded interval has no direct correlated-continuum points.
  if minimum_couplings:
      threshold = min(minimum_couplings)
      ax.axvspan(
          0.0,
          threshold,
          color='gray',
          alpha=.08,
          label=(
              rf'extrapolated region: '
              rf'$0\leq g^2_{{GF}}<{threshold:g}$'
          ),
          zorder=0,
      )

  # Perturbative reference curves.
  xp_pt = np.linspace(0.0, 5.0, 600)
  pt_curves = (
      (1, '-', '1-loop universal'),
      (2, '--', '2-loop universal'),
      (3, '-.', '3-loop gradient flow'),
  )

  for loops, linestyle, label in pt_curves:
      ax.plot(
          xp_pt,
          pt_over_g4(xp_pt, loops),
          color='black',
          ls=linestyle,
          lw=1.4,
          alpha=.65,
          label=label,
          zorder=1,
      )

  # Watermark and analysis information inside the axes.
  ax.text(
      .50,
      .52,
      r'\textbf{Preliminary}',
      transform=ax.transAxes,
      fontsize=42,
      color='gray',
      alpha=.25,
      ha='center',
      va='center',
      rotation=30,
      zorder=20,
      clip_on=True,
  )
  ax.text(
      .58,
      .15,
      (
          FIT_WATERMARK
          + '\n'
          + 'Correlated continuum weak-coupling extrapolation'
          + '\n'
          + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$'
          + '\n'
          + correction_label
          + '\n'
          + r'Bands show posterior $\pm1\sigma$'
          + '\n'
          + r'Fixed $c_0=1$ perturbative limit'
      ),
      transform=ax.transAxes,
      fontsize=10.5,
      color='gray',
      alpha=.82,
      ha='center',
      va='center',
      fontweight='bold',
      zorder=20,
      clip_on=True,
  )

  ax.set(
      xlim=(0, 5),
      ylim=(-0.065, -0.035),
      xlabel=r'$g^2_{GF}$',
      ylabel=r'$\beta_{GF}/g_{GF}^4$',
  )
  ax.legend(
      ncol=2,
      frameon=False,
  )

  save_figure(
      fig,
      window,
      'weak_coupling_to_zero_fit2',
      'correlated',
  )


## Scan manifest and validation summary

In [ ]:
rows=[]
for window in WINDOWS:
    for mode in ('diagonal','correlated'):
        case=load_case(window,mode)
        for operator in OBSERVABLES:
            q=case['cnt_qof'][FLOW][operator]
            rows.append({'tmin':window[0],'tmax':window[1],'mode':mode,'operator':operator,
                         'nfits':len(q),'mean_chi2_dof':np.mean([r['chi2']/r['dof'] for r in q if r['dof']]),
                         'mean_pvalue':np.mean([r['p-value'] for r in q])})
summary=pd.DataFrame(rows)
summary.to_csv(OUTPUT_ROOT/'fit2_scan_summary.csv',index=False)
display(summary)

expected={(window,mode) for window in WINDOWS for mode in ('diagonal','correlated')}
present={(window,mode) for window in WINDOWS for mode in ('diagonal','correlated')
         if (case_dir(window,mode)/'continuum_case.gvar').is_file()}
assert present==expected, f'missing cases: {expected-present}'
assert config.binsize==15 and config.correction=='tln'
assert config.combine=={'s': {'p':5/3,'s':-2/3}}
print('fit2 validation complete:',len(present),'continuum cases')

## Figure 11 integral matching to the perturbative regime

This adapts only the short-window matching idea of Eqs. (8)-(9) and Fig. 11 of [arXiv:2303.00704](https://arxiv.org/abs/2303.00704); the paper studied a different theory, dataset, and quantity. It is not a fit over the full NF4 continuum range. Physical inspection of the correlated NF4 continuum curves motivates the narrow trial window $g^2_{GF}\in[0.8,1.2]$, before their stronger bending develops. The $g^2=0.8$ endpoint is recomputed with the ordinary domain-restricted kernel-correlated continuum procedure. Only this narrow curve fixes the single extra coefficient $b_p$ in $\beta_4(x)=\beta_{\rm PT}^{(3)}(x)-b_p x^5$, with $x=g^2_{GF}$. Repeating the integral match after pointwise $\pm1\sigma$ shifts gives the uncertainty limits. Wilson, clover, and Symanzik results are overlaid on one axes, and their fitted $b_p$ values are printed on the figure.

In [ ]:
from betafn.weak_coupling import (
    continuum_from_extended_interpolants,
    figure11_integral_match,
)

FIG11_MATCH_WINDOW = (0.8, 1.2)
FIG11_MATCH_G2 = np.linspace(*FIG11_MATCH_WINDOW, 9)
FIG11_XMAX = 3.0
fig11_results = {}

for window in WINDOWS:
    case = load_case(window, 'correlated')
    fig, ax = plt.subplots(figsize=(9.5, 6.5))
    fig11_results[window] = {}
    bp_lines = [r'$b_p$ (central; shifted-data range)']

    for operator in OBSERVABLES:
        # Recompute only the narrow matching segment with the standard
        # measured-domain restriction. This supplies the g^2=0.8 endpoint
        # without extrapolating the stored continuum curve from g^2=0.9.
        match_continuum = continuum_from_extended_interpolants(
            bf, FLOW, operator, window, FIG11_MATCH_G2, tau0=config.tau0,
            cov_mode='kernel', kernel='rbf', enforce_domains=True,
        )
        x = np.asarray(case['g2s'][FLOW][operator], dtype=float)
        y = np.asarray(case['betas'][FLOW][operator], dtype=object) / x**2
        result = figure11_integral_match(
            match_continuum['g2'], match_continuum['beta_over_g4'],
            pt_over_g4, match_window=FIG11_MATCH_WINDOW,
        )
        result['matching_continuum'] = match_continuum
        fig11_results[window][operator] = result

        order = np.argsort(x)
        x, y = x[order], y[order]
        shown = x <= FIG11_XMAX
        ax.plot(x[shown], gv.mean(y[shown]), color=OP_COLORS[operator], lw=1.8,
                label=f'{OP_LABELS[operator]} correlated continuum')
        ax.fill_between(x[shown], gv.mean(y[shown])-gv.sdev(y[shown]),
                        gv.mean(y[shown])+gv.sdev(y[shown]),
                        color=OP_COLORS[operator], alpha=.18, linewidth=0)
        mx = match_continuum['g2']
        my = match_continuum['beta_over_g4']
        ax.plot(mx, gv.mean(my), color=OP_COLORS[operator], lw=2.2)
        ax.fill_between(mx, gv.mean(my)-gv.sdev(my), gv.mean(my)+gv.sdev(my),
                        color=OP_COLORS[operator], alpha=.24, linewidth=0)

        xp = np.linspace(0.0, FIG11_MATCH_WINDOW[1], 500)
        central = result['ratio_curve'](xp, 'central')
        shifted_plus = result['ratio_curve'](xp, 'plus_sigma')
        shifted_minus = result['ratio_curve'](xp, 'minus_sigma')
        lower = np.minimum(shifted_plus, shifted_minus)
        upper = np.maximum(shifted_plus, shifted_minus)
        ax.fill_between(xp, lower, upper, color=OP_COLORS[operator],
                        alpha=.12, linewidth=0)
        ax.plot(xp, central, color=OP_COLORS[operator], ls='--', lw=2.1,
                label=rf'{OP_LABELS[operator]} integral-matched $\beta_4$')

        coefficients = result['coefficients']
        coefficient_low = min(coefficients['plus_sigma'], coefficients['minus_sigma'])
        coefficient_high = max(coefficients['plus_sigma'], coefficients['minus_sigma'])
        bp_lines.append(
            f"{OP_LABELS[operator]}: {coefficients['central']:.4g} "
            f"[{coefficient_low:.4g}, {coefficient_high:.4g}]"
        )
        print(window, OP_LABELS[operator], 'Integral-match b_p:', coefficients)

    ax.axvspan(*FIG11_MATCH_WINDOW, facecolor='none', edgecolor='gray',
               hatch='///', lw=0, alpha=.45, label='matching interval')
    xp_pt = np.linspace(0.0, FIG11_XMAX, 500)
    for loops, linestyle, label in (
        (1, '-', '1-loop universal'),
        (2, '--', '2-loop universal'),
        (3, '-.', '3-loop gradient flow'),
    ):
        ax.plot(xp_pt, pt_over_g4(xp_pt, loops), color='gray',
                ls=linestyle, lw=1.2, alpha=.75, label=label)

    ax.text(.50, .53, r'\textbf{Preliminary}', transform=ax.transAxes,
            fontsize=34, color='gray', alpha=.20, ha='center', va='center',
            rotation=30, zorder=20, clip_on=True)
    ax.text(.025, .035, '\n'.join(bp_lines), transform=ax.transAxes,
            fontsize=9.2, ha='left', va='bottom', color='black',
            bbox=dict(boxstyle='round,pad=.35', facecolor='white',
                      edgecolor='0.65', alpha=.88), zorder=30)
    ax.text(.60, .055, FIT_WATERMARK + '\n'
            + rf'Integral match: $g^2\in[{FIG11_MATCH_WINDOW[0]:g},{FIG11_MATCH_WINDOW[1]:g}]$' + '\n'
            + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$, TLN',
            transform=ax.transAxes, fontsize=9.2, color='gray', alpha=.82,
            ha='center', va='center', fontweight='bold', zorder=20)
    ax.set(xlim=(0, FIG11_XMAX), xlabel=r'$g^2_{GF}$',
           ylabel=r'$\beta_{GF}/g_{GF}^4$')
    ax.legend(fontsize=8.2, frameon=False, loc='upper right', ncol=2,
              columnspacing=.8, handlelength=2.2, labelspacing=.35)
    fig.tight_layout()
    save_figure(fig, window, 'figure11_integral_matching_fit2', 'correlated')

## $\Lambda$ parameter at $t_0$

**Blinding:** every reported $\Lambda$ value in this section (GF and $\overline{\mathrm{MS}}$, including the uncertainty envelope) is multiplied by one common hidden factor drawn from $0.6<B<1.4$ with $B\neq1$. The factor is stored outside this notebook and repository and is not printed. Ratios, relative uncertainties, and comparisons among windows, operators, and fit choices are preserved. The numerical scale must not be interpreted until the analysis choices are frozen and the factor is revealed by the blinding custodian.

Following Eq. (2) and Sec. IV of [Phys. Rev. D **108**, 014502 (2023)](https://doi.org/10.1103/PhysRevD.108.014502), the weak-coupling part of the integral uses the matched curve, while the domain-supported part uses the correlated continuum $\beta_{\mathrm{GF}}$. The present Fit1 continuum range reaches the exact $t_0$ coupling for every operator,

$$
g_{\mathrm{GF}}^2(t_0)
=0.3\,\frac{128\pi^2}{3N_c^2-3}
\simeq 15.79
\quad (N_c=3).
$$

The code requires every selected continuum curve to reach this value; it raises an error rather than substituting a lower reference coupling.

For $x=g_{\mathrm{GF}}^2$, the dimensionless GF-scheme result is evaluated from

$$
\frac{\Lambda_{\mathrm{GF}}}{\mu_0}
=
(b_0x_0)^{-b_1/(2b_0^2)}
\exp\!\left[-\frac{1}{2b_0x_0}\right]
\exp\!\left\{-\frac{1}{2}\int_0^{x_0}\!\mathrm{d}x
\left[
\frac{1}{\beta_{\mathrm{GF}}(x)}
+\frac{1}{b_0x^2}
-\frac{b_1}{b_0^2x}
\right]\right\},
\qquad
x_0=g_{\mathrm{GF}}^2(t_0),
\qquad
\mu_0=\frac{1}{\sqrt{8t_0}}.
$$

Thus the two quantities retained in the table are dimensionless; the figure shows the GF-scheme quantity by default:

$$
X_{\mathrm{GF}}(t_0)
\equiv \sqrt{8t_0}\,\Lambda_{\mathrm{GF}}
=\frac{\Lambda_{\mathrm{GF}}}{\mu_0},
\qquad
X_{\overline{\mathrm{MS}}}(t_0)
\equiv \sqrt{8t_0}\,\Lambda_{\overline{\mathrm{MS}}}
=\frac{\Lambda_{\overline{\mathrm{MS}}}}{\mu_0}.
$$

Here $e_1$ is not a fitted coefficient. It is the one-loop perturbative matching coefficient between the GF and $\overline{\mathrm{MS}}$ couplings, defined at $\mu=1/\sqrt{8t}$ by

$$
\alpha_{\mathrm{GF}}
=\alpha_{\overline{\mathrm{MS}}}
\left[1+e_1\frac{\alpha_{\overline{\mathrm{MS}}}}{4\pi}
+\mathcal{O}(\alpha_{\overline{\mathrm{MS}}}^2)\right].
$$

With $\rho=\mu^2t=1/8$, its convention here is

$$
e_1=e_{1,0}+b_0\left[\ln(2\rho)+\gamma_E\right],
\qquad
e_{1,0}=\left(\frac{52}{9}+\frac{22}{3}\ln 2-3\ln 3\right)C_A
-\frac{8}{9}T_F,
$$

where $C_A=N_c=3$, $T_F=T_RN_f=N_f/2=2$, and $b_0=11C_A/3-4T_F/3=25/3$. Therefore $e_1=14.1749604614$ for $N_f=4$. The resulting one-loop GF-to-$\overline{\mathrm{MS}}$ scheme conversion used here is

$$
\frac{\Lambda_{\overline{\mathrm{MS}}}}{\Lambda_{\mathrm{GF}}}
=\exp\!\left[-\frac{e_1}{2b_0}\right]
=0.4272022914,
$$

so that

$$
X_{\overline{\mathrm{MS}}}(t_0)
=0.4272022914\,X_{\mathrm{GF}}(t_0),
\qquad
X_{\mathrm{GF}}(t_0)
=2.340811415\,X_{\overline{\mathrm{MS}}}(t_0).
$$

No value in MeV is inferred in this block. If $\sqrt{8t_0}$ is independently known in fm, either scheme can be converted through

$$
\Lambda_{S}\,[\mathrm{MeV}]
=X_S(t_0)\,
\frac{\hbar c}{\sqrt{8t_0}\,[\mathrm{fm}]},
\qquad
\hbar c=197.3269804\ \mathrm{MeV\,fm},
\qquad
S\in\{\mathrm{GF},\overline{\mathrm{MS}}\}.
$$

The displayed uncertainty is the envelope obtained by repeating the Figure 11 matching with the pointwise central, $+1\sigma$, and $-1\sigma$ continuum curves. It is a diagnostic uncertainty and not yet a complete systematic-error budget.

In [ ]:
from betafn.blinding import add_blinded_watermark, blind_lambda_estimate
from betafn.weak_coupling import lambda_parameter_from_matched_beta

NC = 3
G2_T0 = 0.3 * 128.0 * np.pi**2 / (3.0 * NC**2 - 3.0)
lambda_results = {}
lambda_rows = []

for window in WINDOWS:
    case = load_case(window, 'correlated')
    lambda_results[window] = {}
    for operator in OBSERVABLES:
        x = np.asarray(case['g2s'][FLOW][operator], dtype=float)
        y = np.asarray(case['betas'][FLOW][operator], dtype=object) / x**2
        order = np.argsort(x)
        x, y = x[order], y[order]

        available_g2_max = float(x[-1])
        if available_g2_max < G2_T0 - 1e-10:
            raise RuntimeError(
                f'{window}, {operator}: continuum support ends at ' 
                f'g^2={available_g2_max:g}, below g^2(t0)={G2_T0:g}'
            )
        reference_g2 = G2_T0
        reference_scale = 't0'

        estimate = blind_lambda_estimate(
            lambda_parameter_from_matched_beta(
                x,
                y,
                fig11_results[window][operator],
                bf.perturbative_beta_function,
                reference_g2=reference_g2,
            )
        )
        lambda_results[window][operator] = estimate

        conversion = estimate['lambda_msbar_over_lambda_gf']
        for label in ('central', 'plus_sigma', 'minus_sigma'):
            np.testing.assert_allclose(
                estimate['lambda_msbar_over_mu'][label],
                conversion * estimate['lambda_gf_over_mu'][label],
                rtol=1e-13,
                atol=0.0,
            )

        row = {
            'window': window_tag(window),
            'operator': operator,
            'operator_label': OP_LABELS[operator],
            'reference_scale': reference_scale,
            'g2_reference': reference_g2,
            'g2_supported_max': available_g2_max,
            'g2_t0': G2_T0,
            'reaches_t0': True,
            'lambda_values_blinded': True,
            'lambda_msbar_over_lambda_gf': conversion,
        }
        for scheme, values in (
            ('GF', estimate['lambda_gf_over_mu']),
            ('MSbar', estimate['lambda_msbar_over_mu']),
        ):
            central = values['central']
            envelope_low = min(central, values['plus_sigma'], values['minus_sigma'])
            envelope_high = max(central, values['plus_sigma'], values['minus_sigma'])
            row[f'sqrt8t0_lambda_{scheme}_central'] = central
            row[f'sqrt8t0_lambda_{scheme}_envelope_low'] = envelope_low
            row[f'sqrt8t0_lambda_{scheme}_envelope_high'] = envelope_high
        lambda_rows.append(row)

lambda_table = pd.DataFrame(lambda_rows)
lambda_table.to_csv(OUTPUT_ROOT / 'lambda_parameter_by_window_fit2.csv', index=False)
display(
    lambda_table[[
        'window', 'operator_label', 'reference_scale', 'g2_reference',
        'sqrt8t0_lambda_GF_central',
        'sqrt8t0_lambda_GF_envelope_low',
        'sqrt8t0_lambda_GF_envelope_high',
        'sqrt8t0_lambda_MSbar_central',
        'sqrt8t0_lambda_MSbar_envelope_low',
        'sqrt8t0_lambda_MSbar_envelope_high',
        'lambda_msbar_over_lambda_gf',
    ]].style.format(precision=6)
)

window_labels = [window_tag(window).replace('t_', '').replace('_', '-') for window in WINDOWS]
lambda_schemes = [
    ('GF', r'$\sqrt{8t_0}\,\Lambda_{\rm GF}$'),
]
if PLOT_MSBAR_LAMBDA:
    lambda_schemes.append(
        ('MSbar', r'$\sqrt{8t_0}\,\Lambda_{\overline{\rm MS}}$')
    )

fig, axes = plt.subplots(
    len(lambda_schemes), 1,
    figsize=(12, 5.8 * len(lambda_schemes)),
    sharex=True,
    squeeze=False,
)
axes = axes.ravel()
for ax, (scheme, ylabel) in zip(axes, lambda_schemes):
    for offset, operator in zip((-0.18, 0.0, 0.18), OBSERVABLES):
        subset = lambda_table.query('operator == @operator').set_index('window').loc[
            [window_tag(window) for window in WINDOWS]
        ]
        central = subset[f'sqrt8t0_lambda_{scheme}_central'].to_numpy()
        lower = subset[f'sqrt8t0_lambda_{scheme}_envelope_low'].to_numpy()
        upper = subset[f'sqrt8t0_lambda_{scheme}_envelope_high'].to_numpy()
        ax.errorbar(
            np.arange(len(WINDOWS)) + offset,
            central,
            yerr=np.vstack((central - lower, upper - central)),
            fmt='o',
            capsize=3,
            color=OP_COLORS[operator],
            label=OP_LABELS[operator],
        )
    ax.set_ylabel(ylabel)
    ax.legend(frameon=False, ncol=3)

axes[-1].set_xticks(np.arange(len(WINDOWS)), window_labels, rotation=45, ha='right')
axes[-1].set_xlabel(r'flow-time window $t/a^2$')
fig.suptitle(
    rf'BLINDED $\Lambda$ estimates at ' 
    rf'$g^2_{{\rm GF}}(t_0)={G2_T0:.9f}$'
)
fig.tight_layout()
base = OUTPUT_ROOT / 'lambda_parameter_by_window_fit2'
add_fit_footer(fig)
add_blinded_watermark(fig)
fig.savefig(base.with_suffix('.png'), dpi=300, bbox_inches='tight')
fig.savefig(base.with_suffix('.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

## Diagnostic: extend intermediate interpolations first, then take the continuum limit

This is intentionally separate from both the standard continuum result and the Fig. 11 integral match. The intermediate PT-constrained curves are evaluated down to $g^2=0$ even below their measured coupling domains; at every requested $g^2$, those extrapolated finite-flow-time values are then fitted linearly in $a^2/t$ with the same kernel-correlated weighting used by the standard continuum analysis. The hollow/domain shading marks the region in which this is an interpolation-model extrapolation, so it must be treated as a model-dependence diagnostic rather than additional lattice data.

In [ ]:
from betafn.weak_coupling import continuum_from_extended_interpolants

DIRECT_G2_GRID = np.linspace(0.0, 2.0, 21)
DIRECT_DISPLAY_EPS = 1e-7
direct_weak_results = {}
PLOT_WATERMARK = FIT_WATERMARK.replace(', $x$ errors off', '')

for window in WINDOWS:
    correction_label = ('TLN' if config.correction in ('tln', 'tree-level-normalization')
                        else 'No TLN')

    # First show the intermediate fit curves explicitly continued to g^2=0.
    fig_intermediate, axes_intermediate = plt.subplots(1, 3, figsize=(17, 5.3),
                                                        sharex=True, sharey=True)
    xp = np.linspace(DIRECT_DISPLAY_EPS, DIRECT_G2_GRID[-1], 350)
    for ax, operator in zip(axes_intermediate, OBSERVABLES):
        times = available_flow_times(window)
        for index, time in enumerate(times):
            curve = np.asarray(
                bf.interpolation.model.evaluate(xp, bf.ntrp_fits[FLOW][operator][time]),
                dtype=object,
            ) / xp**2
            domain = tuple(map(float, bf.ntrp_nf[FLOW][operator][time]))
            alpha = .22 + .65 * index / max(len(times)-1, 1)
            ax.plot(xp, gv.mean(curve), color=OP_COLORS[operator], alpha=alpha, lw=1.1)
            ax.axvline(domain[0], color=OP_COLORS[operator], alpha=.10, lw=.7)
        for loops, linestyle, label in (
            (1, '-', '1-loop universal'), (2, '--', '2-loop universal'),
            (3, '-.', '3-loop gradient flow'),
        ):
            ax.plot(xp, pt_over_g4(xp, loops), color='gray', ls=linestyle,
                    lw=1.2, alpha=.75, label=label)
        ax.text(.50, .53, r'\textbf{Preliminary}', transform=ax.transAxes,
                fontsize=29, color='gray', alpha=.23, ha='center', va='center',
                rotation=30, zorder=20, clip_on=True)
        ax.text(.55, .13, PLOT_WATERMARK + '\nExtended intermediate interpolation' + '\n'
                + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$, {correction_label}',
                transform=ax.transAxes, fontsize=9.6, color='gray', alpha=.82,
                ha='center', va='center', fontweight='bold', zorder=20)
        ax.set(xlim=(0, DIRECT_G2_GRID[-1]), xlabel=r'$g^2_{GF}$')
        ax.legend(fontsize=8.8, frameon=False)
    axes_intermediate[0].set_ylabel(r'$\beta_{GF}/g_{GF}^4$')
    save_figure(fig_intermediate, window,
                'extended_intermediate_to_zero_fit2', 'correlated_diagnostic')

    # Then overlay all three operators after taking the correlated continuum limit.
    fig_continuum, ax_continuum = plt.subplots(figsize=(9.5, 6.5))
    direct_weak_results[window] = {}
    supported_from_by_operator = []
    for operator in OBSERVABLES:
        result = continuum_from_extended_interpolants(
            bf, FLOW, operator, window, DIRECT_G2_GRID, tau0=config.tau0,
            cov_mode='kernel', kernel='rbf',
        )
        direct_weak_results[window][operator] = result
        x = result['g2']
        y = result['beta_over_g4']
        supported_from_by_operator.append(
            max(domain[0] for domain in result['measured_domains'].values())
        )
        ax_continuum.plot(x, gv.mean(y), color=OP_COLORS[operator], lw=2.0,
                          label=f'{OP_LABELS[operator]} extended continuum')
        ax_continuum.fill_between(
            x, gv.mean(y)-gv.sdev(y), gv.mean(y)+gv.sdev(y),
            color=OP_COLORS[operator], alpha=.22, linewidth=0,
        )
        print(window, OP_LABELS[operator], 'direct weak continuum QoF:',
              pd.DataFrame(result['quality']).to_string(index=False))
    fully_supported_from = max(supported_from_by_operator)
    ax_continuum.axvspan(
        0, min(fully_supported_from, DIRECT_G2_GRID[-1]),
        color='gray', alpha=.10,
        label='intermediate-fit extrapolation for at least one operator',
    )
    xp_pt = np.linspace(0.0, DIRECT_G2_GRID[-1], 400)
    for loops, linestyle, label in (
        (1, '-', '1-loop universal'), (2, '--', '2-loop universal'),
        (3, '-.', '3-loop gradient flow'),
    ):
        ax_continuum.plot(xp_pt, pt_over_g4(xp_pt, loops), color='gray',
                          ls=linestyle, lw=1.2, alpha=.75, label=label)
    ax_continuum.text(
        .50, .53, r'\textbf{Preliminary}', transform=ax_continuum.transAxes,
        fontsize=34, color='gray', alpha=.23, ha='center', va='center',
        rotation=30, zorder=20, clip_on=True,
    )
    ax_continuum.text(
        .58, .15, PLOT_WATERMARK + '\nExtended-interpolation continuum diagnostic' + '\n'
        + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$, {correction_label}',
        transform=ax_continuum.transAxes, fontsize=10.2, color='gray', alpha=.82,
        ha='center', va='center', fontweight='bold', zorder=20,
    )
    ax_continuum.set(
        xlim=(0, DIRECT_G2_GRID[-1]), ylim=(-0.065, -0.035),
        xlabel=r'$g^2_{GF}$', ylabel=r'$\beta_{GF}/g_{GF}^4$',
    )
    ax_continuum.legend(fontsize=9.2, frameon=False, loc='upper right',
                        ncol=2, columnspacing=.8, handlelength=2.2,
                        labelspacing=.35)
    fig_continuum.tight_layout()
    save_figure(fig_continuum, window,
                'extended_interpolation_continuum_to_zero_fit2',
                'correlated_diagnostic')

## Supplemental continuum-flow-time thinning study

This section is appended to the reference Fit4 analysis; no cell above it is changed. It repeats the original diagonal and kernel-correlated continuum analyses after retaining flow times on five boundary-anchored grids,

$$
\Delta(t/a^2)\in\{0.01,\,0.02,\,0.05,\,0.10,\,0.20\}.
$$

For every spacing and fit window, filled markers are exactly the points used by that continuum fit. At most two hollow markers are added: the valid points nearest one flow-time unit below and above the fit window.

The output layout is the original continuum layout nested inside one directory per thinning spacing:

```text
continuum_thinning_scan/
  run_configuration.json
  continuum_thinning_diagnostics.csv
  continuum_thinning_summary.csv
  dt_0p01/
    selected_flow_times.csv
    t_4_5/
      diagonal/
      correlated/
      diagonal_vs_correlated_fit2.{pdf,png}
    t_5_6/
      ...
    t_4_6/
      ...
  dt_0p02/
    ...
```

Each `diagonal/` and `correlated/` directory receives the same five continuum figures as the unthinned analysis: raw and scaled $a^2/t$ panels, fit quality, and the final raw and scaled continuum curves.


In [ ]:
from contextlib import contextmanager
import json as _json

THINNING_FLOW_SPACINGS = (0.01, 0.02, 0.05, 0.10, 0.20)
THINNING_CONTINUUM_CASES = {
    'diagonal': {
        'cov_mode': 'diagonal', 'diagonal': True,
        'correlated': False, 'error_mode': 'fit',
    },
    'correlated': {
        'cov_mode': 'kernel', 'diagonal': False, 'kernel': 'rbf',
        'correlated': True, 'error_mode': 'fit',
    },
}
THINNING_ROOT = OUTPUT_ROOT / 'continuum_thinning_scan'
THINNING_ROOT.mkdir(parents=True, exist_ok=True)


def thinning_spacing_tag(spacing):
    return f'dt_{spacing:.2f}'.replace('.', 'p')


def thinning_spacing_dir(spacing):
    path = THINNING_ROOT / thinning_spacing_tag(spacing)
    path.mkdir(parents=True, exist_ok=True)
    return path


def thinning_window_dir(spacing, window):
    path = thinning_spacing_dir(spacing) / window_tag(window)
    path.mkdir(parents=True, exist_ok=True)
    return path


def thinning_mode_dir(spacing, window, mode):
    path = thinning_window_dir(spacing, window) / mode
    path.mkdir(parents=True, exist_ok=True)
    return path


def select_thinned_flow_times(available_times, window, spacing):
    '''Choose available fit times nearest a boundary-anchored regular grid.'''
    available = sorted(
        [time for time in available_times
         if window[0] <= float(time) - config.tau0 <= window[1]],
        key=float,
    )
    if not available:
        return []
    targets = list(np.arange(window[0], window[1] + 0.5 * spacing, spacing))
    if not np.isclose(targets[-1], window[1], rtol=0.0, atol=1e-10):
        targets.append(float(window[1]))
    selected = []
    for target in targets:
        nearest = min(
            available,
            key=lambda time: abs((float(time) - config.tau0) - target),
        )
        if nearest not in selected:
            selected.append(nearest)
    for endpoint in (available[0], available[-1]):
        if endpoint not in selected:
            selected.append(endpoint)
    return sorted(selected, key=float)


def _copy_interpolation_subset(container, visible_times):
    subset = {}
    for flow, by_operator in container.items():
        subset[flow] = {}
        for operator, by_time in by_operator.items():
            keep = visible_times.get((flow, operator), tuple(by_time))
            subset[flow][operator] = {
                time: value for time, value in by_time.items() if time in keep
            }
    return subset


@contextmanager
def use_thinned_interpolation_times(window, spacing):
    '''Expose selected fit points plus outside points needed for plotting.'''
    original_fits = bf.interpolation.fits
    original_domains = bf.interpolation.domains
    selected_fit_times = {}
    visible_times = {}
    for flow, by_operator in original_fits.items():
        for operator, by_time in by_operator.items():
            if flow == FLOW and operator in OBSERVABLES:
                selected = select_thinned_flow_times(by_time, window, spacing)
                if len(selected) < 3:
                    raise RuntimeError(
                        f'{window}, {operator}, dt={spacing}: fewer than three '
                        'flow-time points remain for the continuum fit'
                    )
                outside = [
                    time for time in by_time
                    if not window[0]
                    <= float(time) - config.tau0
                    <= window[1]
                ]
                selected_fit_times[(flow, operator)] = tuple(selected)
                visible_times[(flow, operator)] = tuple(selected + outside)
            else:
                selected_fit_times[(flow, operator)] = tuple(by_time)
                visible_times[(flow, operator)] = tuple(by_time)
    bf.interpolation.fits = _copy_interpolation_subset(
        original_fits, visible_times
    )
    bf.interpolation.domains = _copy_interpolation_subset(
        original_domains, visible_times
    )
    bf._assign_stage_aliases()
    try:
        yield selected_fit_times
    finally:
        bf.interpolation.fits = original_fits
        bf.interpolation.domains = original_domains
        bf._assign_stage_aliases()


def snapshot_thinning_case(window, spacing, mode):
    '''Keep one exact continuum result in memory for immediate plotting.'''
    return {
        'window': tuple(window), 'spacing': float(spacing), 'mode': mode,
        'fit_id': FIT_ID, 'order': ORDER,
        'g2s': copy.deepcopy(bf.g2s),
        'betas': copy.deepcopy(bf.betas),
        'cnt_fits': copy.deepcopy(bf.cnt_fits),
        'cnt_qof': copy.deepcopy(bf.continuum.quality),
        'domains': copy.deepcopy(bf.continuum.domains),
        'metadata': copy.deepcopy(bf.continuum.metadata),
    }


def _correlation_diagnostics(covariance):
    covariance = np.asarray(covariance, dtype=float)
    scales = np.sqrt(np.clip(np.diag(covariance), 0.0, None))
    if len(scales) == 0 or np.any(scales <= 0.0):
        return {'condition': np.inf, 'effective_rank': 0.0}
    correlation = covariance / np.outer(scales, scales)
    correlation = 0.5 * (correlation + correlation.T)
    eigenvalues = np.clip(np.linalg.eigvalsh(correlation), 0.0, None)
    denominator = float(np.sum(eigenvalues**2))
    return {
        'condition': float(np.linalg.cond(correlation)),
        'effective_rank': (
            float(np.sum(eigenvalues)**2 / denominator)
            if denominator > 0.0 else 0.0
        ),
    }


def _eligible_full_times(full_fits, full_domains, operator, window, g2):
    return [
        time for time in full_fits[FLOW][operator]
        if float(time) - config.tau0 > 0.0
        and window[0] <= float(time) - config.tau0 <= window[1]
        and full_domains[FLOW][operator][time][0]
        <= g2
        <= full_domains[FLOW][operator][time][-1]
    ]


def collect_thinning_diagnostics(
    window, spacing, mode, options, full_fits, full_domains,
):
    rows = []
    ntrp_eval = bf._wrap_model(bf.ntrp_fcn)
    for operator in OBSERVABLES:
        grid = np.asarray(bf.g2s[FLOW][operator], dtype=float)
        for index, g2 in enumerate(grid):
            fit_input = bf.continuum.fetch(
                'inputs', (FLOW, operator, str(float(g2)))
            )
            times = list(fit_input.labels)
            measured_times = np.asarray([float(time) for time in times])
            nominal_times = measured_times - config.tau0
            jacobian = nominal_times / measured_times
            beta_values = np.asarray([
                factor * ntrp_eval(g2, bf.ntrp_fits[FLOW][operator][time])
                for factor, time in zip(jacobian, times)
            ], dtype=object)
            empirical_covariance = gv.evalcov(beta_values)
            distances = [
                np.subtract.outer(np.log(nominal_times), np.log(nominal_times))
            ]
            weight_covariance = bf._weight_covariance(
                beta_values,
                cov_mode=options['cov_mode'],
                alpha=options.get('alpha', 0.25),
                kernel=options.get('kernel', 'rbf'),
                distances=distances,
            )
            empirical = _correlation_diagnostics(empirical_covariance)
            weighted = _correlation_diagnostics(weight_covariance)
            quality = bf.continuum.quality[FLOW][operator][index]
            full_count = len(_eligible_full_times(
                full_fits, full_domains, operator, window, g2
            ))
            rows.append({
                'window': window_tag(window),
                'spacing': float(spacing),
                'mode': mode,
                'operator': operator,
                'g2': float(g2),
                'n_full': int(full_count),
                'n_retained': int(len(times)),
                'retained_fraction': float(len(times) / full_count),
                'empirical_correlation_condition': empirical['condition'],
                'empirical_effective_rank': empirical['effective_rank'],
                'weight_correlation_condition': weighted['condition'],
                'weight_effective_rank': weighted['effective_rank'],
                'chi2': float(quality['chi2']),
                'dof': int(quality['dof']),
                'chi2_per_dof': (
                    float(quality['chi2'] / quality['dof'])
                    if quality['dof'] else np.nan
                ),
                'p_value': float(quality['p-value']),
            })
    return rows


def _save_thinning_figure(fig, window, spacing, name, mode=None):
    directory = (
        thinning_mode_dir(spacing, window, mode)
        if mode is not None else thinning_window_dir(spacing, window)
    )
    base = directory / name
    footer = globals().get('FIT_FOOTER')
    if footer is not None:
        fig.text(
            0.995, 0.005, footer, ha='right', va='bottom',
            fontsize=8, color='gray', alpha=.75,
        )
    fig.savefig(base.with_suffix('.png'), dpi=300, bbox_inches='tight')
    fig.savefig(base.with_suffix('.pdf'), dpi=300, bbox_inches='tight')
    plt.close(fig)


def plot_thinning_continuum_quality(case, window, spacing, mode):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True)
    for operator in OBSERVABLES:
        grid = np.asarray(case['g2s'][FLOW][operator], dtype=float)
        quality = case['cnt_qof'][FLOW][operator]
        chi = np.asarray([
            row['chi2'] / row['dof'] if row['dof'] else np.nan
            for row in quality
        ], dtype=float)
        pvalues = np.asarray([row['p-value'] for row in quality], dtype=float)
        count = min(len(grid), len(chi), len(pvalues))
        ax1.plot(grid[:count], chi[:count], '-', color=OP_COLORS[operator], lw=1.2)
        ax1.scatter(
            grid[:count], chi[:count], color=OP_COLORS[operator],
            s=28, zorder=3, label=OP_LABELS[operator],
        )
        ax2.plot(grid[:count], pvalues[:count], '-', color=OP_COLORS[operator], lw=1.2)
        ax2.scatter(grid[:count], pvalues[:count], color=OP_COLORS[operator], s=28, zorder=3)
    ax1.axhline(1, color='gray', ls='--')
    ax2.axhline(.05, color='gray', ls='--')
    ax1.set_ylabel(r'$\chi^2/{\rm dof}$')
    ax2.set_ylabel(r'$p$-value')
    ax2.set_xlabel(r'$g^2_{GF}$')
    ax1.legend(frameon=False)
    mode_label = 'Correlated continuum QoF' if mode == 'correlated' else 'Diagonal continuum QoF'
    for ax in (ax1, ax2):
        ax.text(
            .50, .52, 'Preliminary', transform=ax.transAxes,
            fontsize=36, color='gray', alpha=.25, ha='center', va='center',
            rotation=30, zorder=0,
        )
        ax.text(
            .58, .15,
            FIT_WATERMARK + '\n' + mode_label + '\n'
            + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$'
            + '\n' + rf'$\Delta(t/a^2)={spacing:.2f}$',
            transform=ax.transAxes, fontsize=10.5, color='gray', alpha=.92,
            ha='center', va='center', fontweight='bold',
        )
    _save_thinning_figure(
        fig, window, spacing, 'continuum_quality_fit2', mode
    )


def plot_thinning_diagonal_vs_correlated(cases, window, spacing):
    fig, ax = plt.subplots(figsize=(8, 6))
    for operator in OBSERVABLES:
        for mode, linestyle in (('diagonal', '--'), ('correlated', '-')):
            case = cases[mode]
            x = np.asarray(case['g2s'][FLOW][operator], dtype=float)
            y = np.asarray(case['betas'][FLOW][operator], dtype=object) / x**2
            order = np.argsort(x)
            x, y = x[order], y[order]
            mean, sdev = gv.mean(y), gv.sdev(y)
            ax.plot(
                x, mean, color=OP_COLORS[operator], ls=linestyle, lw=1.8,
                label=f'{OP_LABELS[operator]} {mode}',
                zorder=4 if mode == 'correlated' else 3,
            )
            ax.fill_between(
                x, mean - sdev, mean + sdev, color=OP_COLORS[operator],
                alpha=.18 if mode == 'correlated' else .07,
                linewidth=0, zorder=2 if mode == 'correlated' else 1,
            )
    xp = np.linspace(.001, 5.0, 500)
    for loops, linestyle, label in (
        (1, '-', '1-loop universal'),
        (2, '--', '2-loop universal'),
        (3, '-.', '3-loop gradient flow'),
    ):
        ax.plot(
            xp, pt_over_g4(xp, loops), color='gray', ls=linestyle,
            lw=1.5, label=label, zorder=1,
        )
    ax.text(
        .50, .52, 'Preliminary', transform=ax.transAxes,
        fontsize=42, color='gray', alpha=.25, ha='center', va='center',
        rotation=30, zorder=20, clip_on=True,
    )
    ax.text(
        .58, .15,
        FIT_WATERMARK + '\nDiagonal vs correlated continuum\n'
        + rf'$t/a^2\in[{window[0]:g},{window[1]:g}]$'
        + '\n' + rf'$\Delta(t/a^2)={spacing:.2f}$'
        + '\nSolid: correlated; dashed: diagonal',
        transform=ax.transAxes, fontsize=10.5, color='gray', alpha=.82,
        ha='center', va='center', fontweight='bold', zorder=20,
        clip_on=True,
    )
    ax.set(
        xlim=(0, 4.8), xlabel=r'$g^2_{GF}$',
        ylabel=r'$\beta_{GF}/g_{GF}^4$',
    )
    ax.legend(ncol=2, frameon=False)
    _save_thinning_figure(
        fig, window, spacing, 'diagonal_vs_correlated_fit2'
    )


def plot_thinning_case_family(cases, window, spacing):
    '''Reuse the original continuum plot functions with thinning paths.'''
    reference_load_case = globals()['load_case']
    reference_save_figure = globals()['save_figure']
    reference_watermark = globals()['FIT_WATERMARK']
    try:
        globals()['load_case'] = lambda requested_window, mode: cases[mode]
        globals()['save_figure'] = (
            lambda fig, requested_window, name, mode=None:
            _save_thinning_figure(
                fig, requested_window, spacing, name, mode
            )
        )
        globals()['FIT_WATERMARK'] = (
            reference_watermark
            + rf'; thinning $\Delta(t/a^2)={spacing:.2f}$'
        )
        for mode in ('diagonal', 'correlated'):
            plot_continuum_panels(window, mode, divide_by_g4=False)
            plot_continuum_panels(window, mode, divide_by_g4=True)
            plot_final_curve(window, mode, divide_by_g4=False)
            plot_final_curve(window, mode, divide_by_g4=True)
    finally:
        globals()['load_case'] = reference_load_case
        globals()['save_figure'] = reference_save_figure
        globals()['FIT_WATERMARK'] = reference_watermark

    for mode in ('diagonal', 'correlated'):
        plot_thinning_continuum_quality(
            cases[mode], window, spacing, mode
        )
    plot_thinning_diagonal_vs_correlated(cases, window, spacing)


In [ ]:
thinning_rows = []
thinning_selections = []

reference_continuum = bf.continuum
reference_interpolation_fits = bf.interpolation.fits
reference_interpolation_domains = bf.interpolation.domains

try:
    for spacing in THINNING_FLOW_SPACINGS:
        print(f'\n=== thinning {thinning_spacing_tag(spacing)} ===')
        spacing_selections = []
        for window in WINDOWS:
            with use_thinned_interpolation_times(window, spacing) as selections:
                for operator in OBSERVABLES:
                    selected = selections[(FLOW, operator)]
                    selection_row = {
                        'window': window_tag(window),
                        'spacing': float(spacing),
                        'operator': operator,
                        'n_selected_before_domain_restrictions': len(selected),
                        'selected_times': ','.join(map(str, selected)),
                    }
                    thinning_selections.append(selection_row)
                    spacing_selections.append(selection_row)

                cases = {}
                for mode, options in THINNING_CONTINUUM_CASES.items():
                    bf.cnt_xtrp(
                        mnt=window[0], mxt=window[1],
                        mng2=mng2, mxg2=mxg2, dg2=dg2,
                        tau0=config.tau0, v=0,
                        **options,
                    )
                    cases[mode] = snapshot_thinning_case(
                        window, spacing, mode
                    )
                    thinning_rows.extend(collect_thinning_diagnostics(
                        window, spacing, mode, options,
                        reference_interpolation_fits,
                        reference_interpolation_domains,
                    ))
                    print('fitted', thinning_spacing_tag(spacing), window_tag(window), mode)

                plot_thinning_case_family(cases, window, spacing)
                print('saved plots', thinning_spacing_tag(spacing), window_tag(window))
                del cases
                gc.collect()

        pd.DataFrame(spacing_selections).to_csv(
            thinning_spacing_dir(spacing) / 'selected_flow_times.csv',
            index=False,
        )
finally:
    bf.interpolation.fits = reference_interpolation_fits
    bf.interpolation.domains = reference_interpolation_domains
    bf.continuum = reference_continuum
    bf._assign_stage_aliases()

thinning_diagnostics = pd.DataFrame(thinning_rows)
thinning_diagnostics.to_csv(
    THINNING_ROOT / 'continuum_thinning_diagnostics.csv', index=False
)

thinning_summary = (
    thinning_diagnostics
    .groupby(['window', 'mode', 'spacing'], as_index=False)
    .agg(
        fits=('g2', 'size'),
        median_n_retained=('n_retained', 'median'),
        median_retained_fraction=('retained_fraction', 'median'),
        median_empirical_condition=(
            'empirical_correlation_condition', 'median'
        ),
        median_weight_condition=('weight_correlation_condition', 'median'),
        mean_chi2_per_dof=('chi2_per_dof', 'mean'),
        median_p_value=('p_value', 'median'),
    )
)
thinning_summary.to_csv(
    THINNING_ROOT / 'continuum_thinning_summary.csv', index=False
)

configuration = {
    'fit_id': FIT_ID,
    'order': ORDER,
    'fit2_no_priors': FIT_NO_PRIORS,
    'fit2_width': FIT_WIDTH,
    'interpolation_xerrors': config.interpolation.xerrors,
    'flow_time_spacings': list(THINNING_FLOW_SPACINGS),
    'continuum_cases': THINNING_CONTINUUM_CASES,
    'windows': [list(window) for window in WINDOWS],
    'g2_grid': list(G2_GRID),
    'tau0': float(config.tau0),
    'layout': 'dt_<spacing>/t_<window>/{diagonal,correlated}',
    'reference_analysis_modified': False,
}
(THINNING_ROOT / 'run_configuration.json').write_text(
    _json.dumps(configuration, indent=2) + '\n'
)

display(thinning_summary)
print('thinning outputs:', THINNING_ROOT)
